In [ ]:
# Cell 1: Install Required Dependencies and Imports
# This cell sets up all necessary libraries for chunk generation

# Install required packages (if needed)
# !pip install --quiet python-magic

import os
import re
import json
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import hashlib

print("✅ All dependencies imported successfully!")
print("🎯 Sequential Chunk Generation System Ready")

# Saving file for usage

In [ ]:
def create_enhanced_sample_sv_file(filename: str = "RISCV.sv") -> str:
    """Create enhanced sample with more rare regions for testing"""
    enhanced_code = '''module uart_tx #(
    parameter CLK_FREQ = 50_000_000,  // 50 MHz
    parameter BAUD_RATE = 115200
)(
    input  logic       clk,
    input  logic       rst_n,
    input  logic [7:0] tx_data,
    input  logic       tx_valid,
    output logic       tx_ready,
    output logic       tx_out
);

    // Calculate baud rate divisor
    localparam int DIVISOR = CLK_FREQ / BAUD_RATE;
    
    // State machine states
    typedef enum logic [2:0] {
        IDLE,
        START_BIT,
        DATA_BITS,
        STOP_BIT
    } state_t;
    
    state_t state, next_state;
    
    // Internal registers
    logic [7:0] shift_reg;
    logic [2:0] bit_counter;
    logic [$clog2(DIVISOR)-1:0] baud_counter;
    logic baud_tick;
    
    // Baud rate generator
    always_ff @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            baud_counter <= '0;
            baud_tick <= 1'b0;
        end else begin
            if (baud_counter == DIVISOR - 1) begin
                baud_counter <= '0;
                baud_tick <= 1'b1;
            end else begin
                baud_counter <= baud_counter + 1;
                baud_tick <= 1'b0;
            end
        end
    end
    
    // State machine - sequential logic
    always_ff @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            state <= IDLE;
        else
            state <= next_state;
    end
    
    // State machine - combinational logic
    always_comb begin
        next_state = state;
        
        case (state)
            IDLE: begin
                if (tx_valid)
                    next_state = START_BIT;
            end
            
            START_BIT: begin
                if (baud_tick)
                    next_state = DATA_BITS;
            end
            
            DATA_BITS: begin
                if (baud_tick && bit_counter == 7)
                    next_state = STOP_BIT;
            end
            
            STOP_BIT: begin
                if (baud_tick)
                    next_state = IDLE;
            end
            
            default: next_state = IDLE;
        endcase
    end
    
    // Data path logic
    always_ff @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            shift_reg <= '0;
            bit_counter <= '0;
            tx_out <= 1'b1;
            tx_ready <= 1'b1;
        end else begin
            case (state)
                IDLE: begin
                    tx_out <= 1'b1;
                    tx_ready <= 1'b1;
                    if (tx_valid) begin
                        shift_reg <= tx_data;
                        tx_ready <= 1'b0;
                    end
                end
                
                START_BIT: begin
                    tx_out <= 1'b0;  // Start bit
                    bit_counter <= '0;
                end
                
                DATA_BITS: begin
                    if (baud_tick) begin
                        tx_out <= shift_reg[0];
                        shift_reg <= {1'b0, shift_reg[7:1]};
                        bit_counter <= bit_counter + 1;
                    end
                end
                
                STOP_BIT: begin
                    tx_out <= 1'b1;  // Stop bit
                end
                
                default: begin
                    tx_out <= 1'b1;
                    tx_ready <= 1'b1;
                end
            endcase
        end
    end
    
    // Assertions for verification
    `ifdef FORMAL
        // Check that tx_out is only 0 during start bit or data transmission
        assert property (@(posedge clk) disable iff (!rst_n)
            (state == IDLE || state == STOP_BIT) |-> tx_out == 1'b1
        );
        
        // Check that ready signal is low during transmission
        assert property (@(posedge clk) disable iff (!rst_n)
            (state != IDLE) |-> tx_ready == 1'b0
        );
    `endif

endmodule

// Simple testbench
module uart_tx_tb;
    logic clk;
    logic rst_n;
    logic [7:0] tx_data;
    logic tx_valid;
    logic tx_ready;
    logic tx_out;
    
    // Instantiate DUT
    uart_tx #(
        .CLK_FREQ(1_000_000),
        .BAUD_RATE(9600)
    ) dut (.*);
    
    // Clock generation
    initial begin
        clk = 0;
        forever #5 clk = ~clk;
    end
    
    // Test sequence
    initial begin
        rst_n = 0;
        tx_valid = 0;
        tx_data = 8'h00;
        
        #20 rst_n = 1;
        #50;
        
        // Send byte 0x55
        @(posedge clk);
        tx_data = 8'h55;
        tx_valid = 1;
        @(posedge clk);
        tx_valid = 0;
        
        wait(tx_ready);
        #1000;
        
        // Send byte 0xAA
        @(posedge clk);
        tx_data = 8'hAA;
        tx_valid = 1;
        @(posedge clk);
        tx_valid = 0;
        
        wait(tx_ready);
        #1000;
        
        $finish;
    end
    
    // Monitor
    initial begin
        $monitor("Time=%0t rst_n=%b tx_data=%h tx_valid=%b tx_ready=%b tx_out=%b", 
                 $time, rst_n, tx_data, tx_valid, tx_ready, tx_out);
    end

endmodule'''
    
    with open(filename, 'w') as f:
        f.write(enhanced_code)
    
    print(f"✅ Enhanced sample file created: {filename}")
    print(f"📏 Contains multiple rare region types for comprehensive testing")
    return filename

## saving the file 

In [ ]:
create_enhanced_sample_sv_file(filename='RISCV.sv')

# CHUNKING

## Cell 2: Core Chunk Generation Class Definition

In [ ]:
# Cell 2: Core Chunk Generation Class Definition
# This class handles the sequential processing of SystemVerilog/Verilog files

class SequentialChunkGenerator:
    """
    Generates chunks from SystemVerilog/Verilog files in sequential order
    Maintains original line numbers while removing comments for clean content
    """
    
    def __init__(self):
        self.chunk_counter = 0
        self.chunks = {}
        self.file_content = ""
        self.file_lines = []
        self.current_line = 0
        
    def _detect_language(self, filepath: str) -> str:
        """Detect if file is SystemVerilog or Verilog based on extension"""
        ext = os.path.splitext(filepath)[1].lower()
        if ext == '.sv':
            return "SystemVerilog"
        elif ext == '.v':
            return "Verilog"
        else:
            return "Unknown"
    
    def _remove_comments(self, content: str) -> str:
        """
        Remove single-line (//) and multi-line (/* */) comments from code
        Preserves string literals and handles edge cases
        """
        # Remove single-line comments
        lines = content.split('\n')
        cleaned_lines = []
        
        for line in lines:
            # Find // but ignore if inside strings
            in_string = False
            escape_next = False
            comment_pos = -1
            
            for i, char in enumerate(line):
                if escape_next:
                    escape_next = False
                    continue
                    
                if char == '\\':
                    escape_next = True
                elif char == '"' and not escape_next:
                    in_string = not in_string
                elif not in_string and char == '/' and i + 1 < len(line) and line[i + 1] == '/':
                    comment_pos = i
                    break
            
            if comment_pos >= 0:
                line = line[:comment_pos].rstrip()
            
            cleaned_lines.append(line)
        
        content = '\n'.join(cleaned_lines)
        
        # Remove multi-line comments /* */
        content = re.sub(r'/\*.*?\*/', '', content, flags=re.DOTALL)
        
        return content
    
    def _clean_empty_lines(self, content: str) -> str:
        """Remove excessive empty lines while preserving structure"""
        lines = content.split('\n')
        cleaned_lines = []
        prev_empty = False
        
        for line in lines:
            is_empty = line.strip() == ''
            
            if not is_empty:
                cleaned_lines.append(line)
                prev_empty = False
            elif not prev_empty:
                cleaned_lines.append(line)
                prev_empty = True
        
        return '\n'.join(cleaned_lines)

print("✅ SequentialChunkGenerator class defined")
print("🔧 Core functionality: comment removal, language detection, content cleaning")

## Cell 3: Module Detection Methods

In [ ]:
# Cell 3: Module Detection and Boundary Identification
# These methods identify SystemVerilog/Verilog constructs and their boundaries

class SequentialChunkGenerator(SequentialChunkGenerator):
    """Extended with module detection capabilities"""
    
    def _find_module_boundaries(self, content: str) -> List[Dict]:
        """
        Find all module boundaries in sequential order
        Returns list of modules with start/end positions
        """
        modules = []
        lines = content.split('\n')
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            # Check for module declaration
            module_match = re.match(r'module\s+(\w+)', line)
            if module_match:
                module_name = module_match.group(1)
                start_line = i + 1  # 1-based line numbering
                
                # Find corresponding endmodule
                end_line = self._find_endmodule(lines, i)
                
                if end_line:
                    modules.append({
                        'name': module_name,
                        'start_line': start_line,
                        'end_line': end_line,
                        'type': 'module'
                    })
                    i = end_line  # Skip to after this module
                else:
                    i += 1
            else:
                i += 1
        
        return modules
    
    def _find_endmodule(self, lines: List[str], start_idx: int) -> Optional[int]:
        """Find the corresponding endmodule for a module starting at start_idx"""
        module_depth = 1
        
        for i in range(start_idx + 1, len(lines)):
            line = lines[i].strip()
            
            if re.match(r'module\s+\w+', line):
                module_depth += 1
            elif line.startswith('endmodule'):
                module_depth -= 1
                if module_depth == 0:
                    return i + 1  # 1-based line numbering
        
        return None
    
    def _identify_miscellaneous_chunks(self, content: str, modules: List[Dict]) -> List[Dict]:
        """
        Identify non-module chunks (timescale, includes, etc.)
        These are processed in order between modules
        """
        misc_chunks = []
        lines = content.split('\n')
        
        # Find gaps between modules for miscellaneous content
        covered_lines = set()
        for module in modules:
            for line_num in range(module['start_line'], module['end_line'] + 1):
                covered_lines.add(line_num)
        
        # Group uncovered lines into chunks
        current_chunk_start = None
        
        for line_num in range(1, len(lines) + 1):
            if line_num not in covered_lines:
                line_content = lines[line_num - 1].strip()
                
                # Skip completely empty lines
                if line_content:
                    if current_chunk_start is None:
                        current_chunk_start = line_num
            else:
                # End current misc chunk if we hit a module
                if current_chunk_start is not None:
                    misc_chunks.append({
                        'name': f'misc_{current_chunk_start}',
                        'start_line': current_chunk_start,
                        'end_line': line_num - 1,
                        'type': 'miscellaneous'
                    })
                    current_chunk_start = None
        
        # Handle final misc chunk
        if current_chunk_start is not None:
            misc_chunks.append({
                'name': f'misc_{current_chunk_start}',
                'start_line': current_chunk_start,
                'end_line': len(lines),
                'type': 'miscellaneous'
            })
        
        return misc_chunks

print("✅ Module detection methods added")
print("🔍 Capabilities: module boundaries, endmodule detection, miscellaneous chunks")

## Cell 4: Chunk Creation and Processing

In [ ]:
# Cell 4: Chunk Creation and Content Processing (Updated)
# Handles chunk creation with 150-line limit for sub-chunking

class SequentialChunkGenerator(SequentialChunkGenerator):
    """Extended with chunk creation and processing capabilities"""
    
    def __init__(self):
        super().__init__()
        self.remove_comments_enabled = True  # Default: remove comments
    
    def _create_chunk(self, construct: Dict, content: str) -> Dict:
        """
        Create a chunk from a construct (module or miscellaneous)
        Extracts content, removes comments, and calculates metadata
        """
        self.chunk_counter += 1
        chunk_id = f"chunk_{self.chunk_counter:03d}"
        
        # Extract content for this chunk
        lines = content.split('\n')
        start_idx = construct['start_line'] - 1  # Convert to 0-based
        end_idx = construct['end_line'] - 1      # Convert to 0-based
        
        chunk_lines = lines[start_idx:end_idx + 1]
        raw_content = '\n'.join(chunk_lines)
        
        # Clean content (remove comments if enabled)
        if self.remove_comments_enabled:
            cleaned_content = self._remove_comments(raw_content)
            cleaned_content = self._clean_empty_lines(cleaned_content)
        else:
            cleaned_content = raw_content
        
        # Calculate actual content lines (non-empty after cleaning)
        content_lines = [line for line in cleaned_content.split('\n') if line.strip()]
        
        chunk = {
            "chunk_id": chunk_id,
            "start_line": construct['start_line'],
            "end_line": construct['end_line'],
            "content": cleaned_content,
            "parent_module": None,
            "parent_construct": construct.get('name', None),
            "is_sub_chunk": False,
            "parent_chunk": None,
            "line_count": len(content_lines),
            "construct_type": construct['type']
        }
        
        return chunk_id, chunk
    
    def _should_split_module(self, chunk: Dict) -> bool:
        """
        Determine if a module should be split into sub-chunks
        Split only if module is larger than 150 lines
        """
        return chunk['construct_type'] == 'module' and chunk['line_count'] > 150
    
    def _create_sub_chunks(self, parent_chunk_id: str, parent_chunk: Dict) -> Dict:
        """
        Split large modules into smaller sub-chunks (a, b, c, etc.)
        Maintains parent-child relationships
        """
        sub_chunks = {}
        content = parent_chunk['content']
        lines = content.split('\n')
        content_lines = [line for line in lines if line.strip()]
        
        # Calculate how many sub-chunks we need
        total_content_lines = len(content_lines)
        chunks_needed = max(2, (total_content_lines + 149) // 150)  # Ceiling division
        
        print(f"     📦 Splitting {total_content_lines} lines into {chunks_needed} sub-chunks")
        
        # Find good split points
        split_points = []
        chunk_size = len(lines) // chunks_needed
        
        for i in range(1, chunks_needed):
            target_split = i * chunk_size
            actual_split = self._find_good_split_point(lines, target_split)
            split_points.append(actual_split)
        
        # Create sub-chunks
        sub_chunk_labels = ['a', 'b', 'c', 'd', 'e']  # Support up to 5 sub-chunks
        prev_split = 0
        
        for i in range(chunks_needed):
            sub_chunk_id = f"{parent_chunk_id}_{sub_chunk_labels[i]}"
            
            # Determine end point
            if i == chunks_needed - 1:
                # Last chunk gets everything remaining
                end_split = len(lines)
            else:
                end_split = split_points[i]
            
            # Extract content for this sub-chunk
            sub_chunk_lines = lines[prev_split:end_split]
            sub_chunk_content = '\n'.join(sub_chunk_lines)
            sub_content_lines = [line for line in sub_chunk_lines if line.strip()]
            
            sub_chunk = {
                "chunk_id": sub_chunk_id,
                "start_line": parent_chunk['start_line'] + prev_split,
                "end_line": parent_chunk['start_line'] + end_split - 1,
                "content": sub_chunk_content,
                "parent_module": None,
                "parent_construct": parent_chunk['parent_construct'],
                "is_sub_chunk": True,
                "parent_chunk": parent_chunk_id,
                "line_count": len(sub_content_lines),
                "construct_type": parent_chunk['construct_type']
            }
            
            sub_chunks[sub_chunk_id] = sub_chunk
            print(f"       └─ {sub_chunk_id}: {len(sub_content_lines)} content lines")
            
            prev_split = end_split
        
        return sub_chunks
    
    def _find_good_split_point(self, lines: List[str], target_line: int) -> int:
        """Find a logical place to split the module (after semicolon, end, etc.)"""
        search_range = 20  # Look within 20 lines of target
        
        # Preferred split patterns (in order of preference)
        good_split_patterns = [
            r'^\s*end\s*$',                    # 'end' statements
            r'^\s*endmodule\s*$',              # 'endmodule' 
            r';\s*$',                          # Lines ending with semicolon
            r'^\s*\)\s*;\s*$',                 # Closing parenthesis with semicolon
            r'^\s*end\w+\s*$',                 # end something (endcase, etc.)
        ]
        
        # First try to find good patterns near target
        for offset in range(search_range):
            for line_idx in [target_line + offset, target_line - offset]:
                if 0 <= line_idx < len(lines):
                    line = lines[line_idx].strip()
                    
                    for pattern in good_split_patterns:
                        if re.search(pattern, line):
                            return line_idx + 1
        
        # Fallback: find any non-empty line near target
        for offset in range(search_range):
            for line_idx in [target_line + offset, target_line - offset]:
                if 0 <= line_idx < len(lines):
                    if lines[line_idx].strip():
                        return line_idx + 1
        
        # Final fallback to target line
        return max(1, min(target_line, len(lines)))

print("✅ Chunk creation methods updated")
print("🎯 Features: 150-line limit, smart sub-chunking, optional comment removal")

## Cell 5: Main Processing Pipeline

In [ ]:
# Cell 5: Main Processing Pipeline
# The core method that orchestrates the entire chunk generation process

class SequentialChunkGenerator(SequentialChunkGenerator):
    """Complete chunk generator with main processing pipeline"""
    
    def generate_chunks(self, filepath: str) -> Dict:
        """
        Main method to generate chunks from a SystemVerilog/Verilog file
        Processes file sequentially to maintain order
        
        Args:
            filepath: Path to the .sv or .v file
            
        Returns:
            Dictionary containing file metadata, chunks, and statistics
        """
        print(f"🚀 Starting chunk generation for: {os.path.basename(filepath)}")
        
        # Reset state for new file
        self.chunk_counter = 0
        self.chunks = {}
        
        # Read file
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                self.file_content = f.read()
        except Exception as e:
            raise Exception(f"Failed to read file {filepath}: {e}")
        
        self.file_lines = self.file_content.split('\n')
        
        # Generate file metadata
        file_stats = os.stat(filepath)
        file_metadata = {
            "filename": os.path.basename(filepath),
            "filepath": filepath,
            "total_lines": len(self.file_lines),
            "language": self._detect_language(filepath),
            "file_size_bytes": file_stats.st_size,
            "created_at": datetime.now().isoformat()
        }
        
        print(f"📁 File info: {file_metadata['total_lines']} lines, {file_metadata['language']}")
        
        # Find all constructs in sequential order
        modules = self._find_module_boundaries(self.file_content)
        misc_chunks = self._identify_miscellaneous_chunks(self.file_content, modules)
        
        # Combine and sort all constructs by start_line
        all_constructs = modules + misc_chunks
        all_constructs.sort(key=lambda x: x['start_line'])
        
        print(f"🔍 Found {len(modules)} modules and {len(misc_chunks)} misc sections")
        
        # Process constructs sequentially
        chunk_sequence = []
        sub_chunks_created = 0
        
        for construct in all_constructs:
            print(f"   Processing {construct['type']}: {construct.get('name', 'misc')} (lines {construct['start_line']}-{construct['end_line']})")
            
            # Create main chunk
            chunk_id, chunk = self._create_chunk(construct, self.file_content)
            self.chunks[chunk_id] = chunk
            chunk_sequence.append(chunk_id)
            
            # Check if we need to create sub-chunks
            if self._should_split_module(chunk):
                print(f"     📦 Large module ({chunk['line_count']} lines) - creating sub-chunks")
                
                # Remove the parent chunk from main chunks
                del self.chunks[chunk_id]
                chunk_sequence.remove(chunk_id)
                
                # Create sub-chunks
                sub_chunks = self._create_sub_chunks(chunk_id, chunk)
                
                # Add sub-chunks to main collection
                for sub_chunk_id, sub_chunk in sub_chunks.items():
                    self.chunks[sub_chunk_id] = sub_chunk
                    chunk_sequence.append(sub_chunk_id)
                
                sub_chunks_created += len(sub_chunks)
        
        # Generate statistics
        total_content_lines = sum(chunk['line_count'] for chunk in self.chunks.values())
        
        chunking_stats = {
            "total_chunks": len(self.chunks),
            "sub_chunks": sub_chunks_created,
            "average_chunk_size": total_content_lines / len(self.chunks) if self.chunks else 0,
            "constructs_preserved": len(modules),
            "comments_removed": True
        }
        
        print(f"✅ Generated {len(self.chunks)} chunks ({sub_chunks_created} sub-chunks)")
        print(f"📊 Average chunk size: {chunking_stats['average_chunk_size']:.1f} lines")
        
        # Return complete chunk data
        return {
            "file_metadata": file_metadata,
            "chunks": self.chunks,
            "chunk_sequence": chunk_sequence,
            "chunking_stats": chunking_stats
        }

print("✅ Main processing pipeline complete")
print("🎯 Ready to process SystemVerilog/Verilog files sequentially")

## Cell 6: Utility Functions

In [ ]:
# Cell 6: Utility Functions
# Helper functions for file operations and chunk validation

def save_chunks_to_file(chunks_data: Dict, output_path: str) -> str:
    """
    Save chunk data to JSON file with proper formatting
    
    Args:
        chunks_data: Dictionary containing chunk information
        output_path: Path where to save the JSON file
        
    Returns:
        Path to saved file
    """
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(chunks_data, f, indent=2, ensure_ascii=False)
        
        print(f"💾 Chunks saved to: {output_path}")
        return output_path
        
    except Exception as e:
        print(f"❌ Failed to save chunks: {e}")
        raise

def generate_chunks_for_file(input_filepath: str, output_dir: str = None) -> str:
    """
    Complete workflow to generate chunks for a single file
    
    Args:
        input_filepath: Path to input .sv or .v file
        output_dir: Directory to save output (default: same as input)
        
    Returns:
        Path to generated chunks file
    """
    # Determine output path
    if output_dir is None:
        output_dir = os.path.dirname(input_filepath)
    
    base_name = os.path.splitext(os.path.basename(input_filepath))[0]
    output_path = os.path.join(output_dir, f"{base_name}_chunks.json")
    
    # Generate chunks
    generator = SequentialChunkGenerator()
    chunks_data = generator.generate_chunks(input_filepath)
    
    # Save to file
    save_chunks_to_file(chunks_data, output_path)
    
    return output_path

def validate_chunks(chunks_data: Dict) -> bool:
    """
    Basic validation of chunk data structure
    
    Args:
        chunks_data: Chunk data dictionary
        
    Returns:
        True if valid, False otherwise
    """
    try:
        # Check required top-level keys
        required_keys = ['file_metadata', 'chunks', 'chunk_sequence', 'chunking_stats']
        for key in required_keys:
            if key not in chunks_data:
                print(f"❌ Missing required key: {key}")
                return False
        
        # Check chunk sequence matches chunks
        chunk_ids = set(chunks_data['chunks'].keys())
        sequence_ids = set(chunks_data['chunk_sequence'])
        
        if chunk_ids != sequence_ids:
            print(f"❌ Chunk sequence mismatch")
            print(f"   Chunks: {sorted(chunk_ids)}")
            print(f"   Sequence: {sorted(sequence_ids)}")
            return False
        
        # Check chunk line numbers are sequential
        chunks = chunks_data['chunks']
        sequence = chunks_data['chunk_sequence']
        
        prev_end_line = 0
        for chunk_id in sequence:
            chunk = chunks[chunk_id]
            if chunk['start_line'] <= prev_end_line:
                # Allow some overlap for sub-chunks
                if not chunk.get('is_sub_chunk', False):
                    print(f"❌ Chunk {chunk_id} has invalid line ordering")
                    return False
            prev_end_line = chunk['end_line']
        
        print("✅ Chunk validation passed")
        return True
        
    except Exception as e:
        print(f"❌ Validation error: {e}")
        return False

print("✅ Utility functions ready")
print("🛠️ Features: file saving, workflow automation, validation")

## Cell 8: Simplified Interactive Chunk Generator

In [ ]:
# Cell 8: Simplified Interactive Chunk Generator (Updated)
# Simple interface with 150-line sub-chunking and comment removal option

def simplified_chunk_generator():
    """
    Simplified chunk generator with 150-line limit
    Automatically saves with filename_chunks.json naming
    Shows first chunk as preview
    """
    print("🎯 Simplified Chunk Generator (150-line limit)")
    print("=" * 55)
    
    # Get input file path
    input_file = input("📁 Enter file path: ").strip()
    
    # Check if file exists
    if not os.path.exists(input_file):
        print(f"❌ File not found: {input_file}")
        return None
    
    # Ask about comment removal (default True)
    remove_comments_input = input("🧹 Remove comments? (Y/n, default=Y): ").strip().lower()
    remove_comments = remove_comments_input != 'n'
    
    print(f"   Comment removal: {'✅ Enabled' if remove_comments else '❌ Disabled'}")
    print(f"   Sub-chunk limit: 150 lines per chunk")
    
    # Generate output filename automatically
    base_name = os.path.splitext(os.path.basename(input_file))[0]
    output_dir = "/kaggle/working"
    output_file = os.path.join(output_dir, f"{base_name}_chunks.json")
    
    # Process file
    try:
        print(f"\n🚀 Processing: {os.path.basename(input_file)}")
        print(f"💾 Output will be saved as: {os.path.basename(output_file)}")
        
        # Create generator with comment removal option
        generator = SequentialChunkGenerator()
        generator.remove_comments_enabled = remove_comments
        
        chunks_data = generator.generate_chunks(input_file)
        
        # Save chunks
        save_chunks_to_file(chunks_data, output_file)
        
        # Display summary
        print(f"\n📊 Processing Complete!")
        print(f"   ✅ File: {chunks_data['file_metadata']['filename']}")
        print(f"   ✅ Language: {chunks_data['file_metadata']['language']}")
        print(f"   ✅ Total lines: {chunks_data['file_metadata']['total_lines']}")
        print(f"   ✅ Total chunks: {len(chunks_data['chunks'])}")
        print(f"   ✅ Sub-chunks: {chunks_data['chunking_stats']['sub_chunks']}")
        print(f"   ✅ Output saved: {output_file}")
        
        # Show chunk breakdown
        print(f"\n📋 Chunk Breakdown:")
        print("-" * 40)
        for i, chunk_id in enumerate(chunks_data['chunk_sequence'], 1):
            chunk = chunks_data['chunks'][chunk_id]
            sub_indicator = " (sub)" if chunk.get('is_sub_chunk', False) else ""
            print(f"   {i:2d}. {chunk_id}{sub_indicator}: lines {chunk['start_line']}-{chunk['end_line']} ({chunk['line_count']} lines)")
        
        # Show first chunk as preview
        print(f"\n📋 First Chunk Preview:")
        print("=" * 50)
        
        first_chunk_id = chunks_data['chunk_sequence'][0]
        first_chunk = chunks_data['chunks'][first_chunk_id]
        
        print(f"Chunk ID: {first_chunk_id}")
        print(f"Lines: {first_chunk['start_line']}-{first_chunk['end_line']} ({first_chunk['line_count']} content lines)")
        print(f"Type: {first_chunk['construct_type']}")
        print(f"Construct: {first_chunk['parent_construct'] or 'N/A'}")
        print(f"Sub-chunk: {'Yes' if first_chunk.get('is_sub_chunk', False) else 'No'}")
        print("\nContent (first 15 lines):")
        print("-" * 30)
        
        content_lines = first_chunk['content'].split('\n')
        for i, line in enumerate(content_lines[:15], 1):
            print(f"{i:2d}: {line}")
        
        if len(content_lines) > 15:
            print(f"... ({len(content_lines) - 15} more lines)")
        
        print("-" * 30)
        
        # Quick validation
        is_valid = validate_chunks(chunks_data)
        print(f"\n✅ Validation: {'PASSED' if is_valid else 'FAILED'}")
        
        return chunks_data
        
    except Exception as e:
        print(f"❌ Error during processing: {e}")
        return None

# Ready to use
print("🎯 Ready to run: simplified_chunk_generator()")
print("   • 150-line limit for sub-chunking")
print("   • Module-based chunking strategy")
print("   • Optional comment removal (default: Yes)")
print("   • Sequential ordering guaranteed")
print("\n🚀 Run: simplified_chunk_generator()")

## Usage

In [ ]:
# Replace with your actual file path
input_file = "/kaggle/working/RISCV.sv"

# This creates filename_chunks.json automatically
output_file = generate_chunks_for_file(input_file)

print(f"✅ Chunks saved to: {output_file}")

## Signal Extraction with Declaration Lines

In [ ]:
import json
import re
from typing import Dict, List, Optional
from collections import defaultdict

class ModuleSignalExtractor:
    """
    Extract all signals from SystemVerilog modules with declaration line numbers.
    Interactive GPT-based extraction for accuracy.
    """
    
    def __init__(self):
        self.module_signals = {}
    
    def load_chunks(self, chunks_file: str):
        """Load chunk data."""
        with open(chunks_file, 'r') as f:
            self.chunks_data = json.load(f)
        print(f"Loaded {len(self.chunks_data['chunks'])} chunks")
    
    def group_chunks_by_module(self) -> Dict[str, List[str]]:
        """Group chunks by parent module."""
        module_groups = defaultdict(list)
        
        for chunk_id in sorted(self.chunks_data['chunks'].keys()):
            parts = chunk_id.split('_')
            if len(parts) >= 2:
                module_id = f"{parts[0]}_{parts[1]}"
            else:
                module_id = chunk_id
            
            module_groups[module_id].append(chunk_id)
        
        return dict(module_groups)
    
    def get_combined_module_code(self, chunk_ids: List[str]) -> str:
        """Combine all chunks for a module into single code block."""
        lines = []
        for chunk_id in chunk_ids:
            chunk = self.chunks_data['chunks'][chunk_id]
            lines.append(f"// Chunk: {chunk_id} (Lines {chunk['start_line']}-{chunk['end_line']})")
            lines.append(chunk['content'])
            lines.append("")
        return '\n'.join(lines)
    
    def generate_signal_extraction_prompt(self, module_id: str, chunk_ids: List[str]) -> str:
        """Generate GPT prompt for signal extraction."""
        
        first_chunk = self.chunks_data['chunks'][chunk_ids[0]]
        module_code = self.get_combined_module_code(chunk_ids)
        
        module_name = "unknown"
        match = re.search(r'module\s+(\w+)', first_chunk['content'])
        if match:
            module_name = match.group(1)
        
        prompt = f"""SYSTEMVERILOG SIGNAL EXTRACTION ANALYSIS

=== TASK ===
Extract ALL signal declarations from this SystemVerilog module.
Identify signal types, widths, and DECLARATION LINE NUMBERS.

=== MODULE INFO ===
Module ID: {module_id}
Module Name: {module_name}
Chunks: {', '.join(chunk_ids)}

=== MODULE CODE ===
{module_code}

=== EXTRACTION REQUIREMENTS ===
Extract the following with DECLARATION LINE NUMBERS:

1. MODULE PORTS (inputs/outputs):
   - Signal name
   - Direction (input/output/inout)
   - Type (logic/wire/reg)
   - Width (if any)
   - DECLARATION LINE NUMBER (absolute line from code above)

2. MODULE-LEVEL SIGNALS (wire/reg/logic):
   - Signal name
   - Type (wire/reg/logic)
   - Width (if any)
   - DECLARATION LINE NUMBER
   - Scope: always "module"

3. PARAMETERS:
   - Parameter name
   - Type (parameter/localparam)
   - Value (if specified)
   - DECLARATION LINE NUMBER

4. TYPEDEFS & ENUMS:
   - Type name
   - Definition
   - Possible values (for enums)
   - DECLARATION LINE NUMBER

5. STATE MACHINE SIGNALS (if any):
   - State variable name
   - Type (enum name)
   - DECLARATION LINE NUMBER

=== OUTPUT FORMAT (JSON ONLY) ===
{{
  "module_id": "{module_id}",
  "module_name": "{module_name}",
  "chunks": {chunk_ids},
  "ports": [
    {{"name": "clk", "direction": "input", "type": "logic", "width": "", "declared_at_line": 5}},
    {{"name": "tx_data", "direction": "input", "type": "logic", "width": "[7:0]", "declared_at_line": 7}}
  ],
  "module_signals": [
    {{"name": "shift_reg", "type": "logic", "width": "[7:0]", "declared_at_line": 20, "scope": "module"}},
    {{"name": "baud_counter", "type": "logic", "width": "[$clog2(DIVISOR)-1:0]", "declared_at_line": 21, "scope": "module"}}
  ],
  "parameters": [
    {{"name": "CLK_FREQ", "type": "parameter", "value": "50_000_000", "declared_at_line": 2}},
    {{"name": "DIVISOR", "type": "localparam", "value": "CLK_FREQ / BAUD_RATE", "declared_at_line": 15}}
  ],
  "enums": [
    {{
      "name": "state_t",
      "type": "enum logic [2:0]",
      "values": ["IDLE", "START_BIT", "DATA_BITS", "STOP_BIT"],
      "declared_at_line": 13
    }}
  ],
  "state_signals": [
    {{"name": "state", "type": "state_t", "declared_at_line": 23}},
    {{"name": "next_state", "type": "state_t", "declared_at_line": 24}}
  ]
}}

=== CRITICAL: DECLARATION LINE NUMBERS ===
- Use the ABSOLUTE line numbers from the code shown above
- This is essential for scope validation in trojan implementation
- Ports and parameters are accessible from anywhere (mark their lines anyway)
- Module signals are accessible from their declaration line onwards

=== IMPORTANT NOTES ===
- Include ALL signals declared in the module
- Use exact line numbers from the code comments
- For signals with complex widths (like $clog2), include the full expression
- Skip local variables inside always blocks
- Focus on module-level declarations only

Output JSON now:"""
        
        return prompt
    
    def parse_signal_extraction_response(self, gpt_response: str, module_id: str) -> Optional[Dict]:
        """Parse GPT signal extraction response."""
        response = gpt_response.strip()
        
        if '```json' in response:
            response = response.split('```json')[1].split('```')[0].strip()
        elif '```' in response:
            response = response.split('```')[1].split('```')[0].strip()
        
        try:
            signal_data = json.loads(response)
            self.module_signals[module_id] = signal_data
            return signal_data
        except json.JSONDecodeError as e:
            print(f"JSON parse error: {e}")
            return None
    
    def run_signal_extraction(self, chunks_file: str, output_file: str = None):
        """Run interactive signal extraction for all modules."""
        print("="*70)
        print("MODULE SIGNAL EXTRACTION WITH DECLARATION LINES")
        print("="*70)
        
        self.load_chunks(chunks_file)
        module_groups = self.group_chunks_by_module()
        
        print(f"\nFound {len(module_groups)} unique modules:")
        for module_id, chunks in module_groups.items():
            print(f"   {module_id}: {len(chunks)} chunk(s)")
        
        print(f"\n{'='*70}")
        print("EXTRACTING SIGNALS")
        print("="*70)
        
        for idx, (module_id, chunk_ids) in enumerate(module_groups.items(), 1):
            print(f"\n{'='*70}")
            print(f"MODULE {idx}/{len(module_groups)}: {module_id}")
            print(f"{'='*70}")
            
            prompt = self.generate_signal_extraction_prompt(module_id, chunk_ids)
            
            print(f"\n{'─'*70}")
            print("GPT PROMPT:")
            print("─"*70)
            print(prompt)
            print("─"*70)
            
            print("\nINSTRUCTIONS:")
            print("1. Copy the prompt above")
            print("2. Paste into ChatGPT/Claude")
            print("3. Copy the JSON response")
            print("4. Paste below")
            print("Type 'skip' to skip, 'quit' to exit")
            
            while True:
                user_input = input(f"\nPaste signal data for {module_id}: ").strip()
                
                if user_input.lower() == 'quit':
                    print("Terminated")
                    return self.module_signals
                
                if user_input.lower() == 'skip':
                    print("Skipped")
                    break
                
                if len(user_input) < 20:
                    print("Response too short. Paste complete JSON.")
                    continue
                
                signal_result = self.parse_signal_extraction_response(user_input, module_id)
                
                if signal_result:
                    print(f"\nSignals extracted for {module_id}:")
                    print(f"   Ports: {len(signal_result.get('ports', []))}")
                    print(f"   Module Signals: {len(signal_result.get('module_signals', []))}")
                    print(f"   Parameters: {len(signal_result.get('parameters', []))}")
                    print(f"   Enums: {len(signal_result.get('enums', []))}")
                    break
                else:
                    print("Parse failed. Try again or type 'skip'")
        
        if output_file is None:
            output_file = chunks_file.replace('_chunks.json', '_signals.json')
        
        output_data = {
            'metadata': {
                'source_chunks_file': chunks_file,
                'total_modules': len(module_groups),
                'modules_extracted': len(self.module_signals)
            },
            'module_signals': self.module_signals
        }
        
        with open(output_file, 'w') as f:
            json.dump(output_data, f, indent=2)
        
        print(f"\n{'='*70}")
        print("SIGNAL EXTRACTION COMPLETE")
        print("="*70)
        print(f"Results saved: {output_file}")
        print(f"Modules processed: {len(self.module_signals)}/{len(module_groups)}")
        
        return self.module_signals


def extract_module_signals(chunks_file: str, output_file: str = None):
    """
    Extract signals from all modules in chunks file.
    
    Args:
        chunks_file: Path to chunks JSON file
        output_file: Output path (default: filename_signals.json)
    """
    extractor = ModuleSignalExtractor()
    return extractor.run_signal_extraction(chunks_file, output_file)


print("MODULE SIGNAL EXTRACTOR READY")
print("="*60)
print("\nUsage:")
print("  extract_module_signals('/kaggle/working/RISCV_chunks.json')")
print("\nThis creates: RISCV_signals.json")
print("Signal metadata includes declaration line numbers for scope validation")

In [ ]:
extract_module_signals('/kaggle/working/RISCV_chunks.json')

# Rare region finding

## Cell 9: Rare Region Detection with Chunk Mapping

In [ ]:
# Cell 9: Simplified Rare Region Detection with Exact Code Blocks
# Clean prompt + robust multi-line code matching

import json
import pandas as pd
import re
from datetime import datetime

def interactive_analysis(chunks_file):
    """
    Interactive analysis to create rare regions JSON with chunk location mapping.
    Simple prompt: Get EXACT VERBATIM code blocks from GPT.
    """
    # Load chunks
    with open(chunks_file, 'r') as f:
        chunks_data = json.load(f)
    
    all_rare_regions = []
    chunks_with_regions = 0
    total_chunks = len(chunks_data['chunks'])
    
    print(f"🚀 Starting analysis of {total_chunks} chunks...")
    print("=" * 60)
    
    for chunk_id, chunk in chunks_data['chunks'].items():
        print(f"\n🧩 CHUNK: {chunk_id}")
        print("🏗️ TYPE:", chunk.get('construct_type', 'unknown'), "| 📦 MODULE:", chunk.get('parent_module', 'None'))
        
        print("\nTASK: Analyze this SystemVerilog code and identify RARE EXECUTION REGIONS.")
        print("\nLOOK FOR any code patterns that execute INFREQUENTLY during normal operation:")
        print("• Conditional blocks that trigger only under specific/uncommon conditions")
        print("• Error handling paths and exception cases")
        print("• Initialization, reset, or startup sequences")
        print("• Debug, test, or configuration modes")
        print("• Complex loops or computationally expensive operations")
        print("• Security checks, privilege validations, or audit trails")
        print("• Cache miss handlers, memory fault recovery")
        print("• Calibration routines, self-test procedures")
        print("• Interrupt handlers for rare events")
        print("• Fallback mechanisms or recovery procedures")
        print("• Any 'if' conditions that rarely evaluate to true")
        print("• Code paths triggered only during specific system states")
        
        print("\nCODE TO ANALYZE:")
        print("─" * 40)
        print(chunk['content'])
        print("─" * 40)
        
        # SIMPLE AND CLEAR PROMPT
        print("\n" + "=" * 60)
        print("📋 CODE SNIPPET REQUIREMENTS:")
        print("=" * 60)
        print("• Provide the EXACT multi-line code block as it appears in the chunk")
        print("• Copy it VERBATIM - no paraphrasing, no abbreviation, no ellipsis (...)")
        print("• Include ALL lines that constitute the rare region")
        print("\n📋 REQUIRED JSON OUTPUT FORMAT:")
        print('[')
        print('  {')
        print('    "rare_region_type": "HAZARD_DETECTION",')
        print('    "code_snippet": "wire load_use_hazard = idex_memread & idex_valid & ifid_valid & (\\n    (id_use_rs1 && (idex_rd != 5\'d0) && (idex_rd == id_rs1)) ||\\n    (id_use_rs2 && (idex_rd != 5\'d0) && (idex_rd == id_rs2))\\n);",')
        print('    "activation_trigger": "Load-use hazard condition",')
        print('    "confidence_score": 0.9,')
        print('    "description": "Pipeline hazard detection requiring stall"')
        print('  }')
        print(']')
        print("\n💡 If NO rare regions found, output: []")
        print("=" * 60)
        
        # Collect GPT response
        print(f"\n📋 Paste GPT response for {chunk_id} (type 'DONE' on new line when finished):")
        gpt_response = []
        while True:
            line = input()
            if line.strip() == 'DONE':
                break
            gpt_response.append(line)
        
        # Process response and find locations in chunk
        response_text = '\n'.join(gpt_response)
        chunk_regions = process_gpt_response_with_location(response_text, chunk_id, chunk)
        
        if chunk_regions:
            all_rare_regions.extend(chunk_regions)
            chunks_with_regions += 1
            print(f"✅ Found {len(chunk_regions)} rare regions in {chunk_id}")
        else:
            print(f"ℹ️ No rare regions found in {chunk_id}")
    
    # Create final JSON
    output_file = chunks_file.replace('_chunks.json', '_rare_regions.json')
    result = {
        "file_metadata": {
            "source_file": chunks_data['file_metadata']['filename'],
            "total_chunks_analyzed": total_chunks,
            "chunks_with_rare_regions": chunks_with_regions,
            "total_rare_regions": len(all_rare_regions),
            "analysis_timestamp": datetime.now().isoformat()
        },
        "rare_regions": all_rare_regions
    }
    
    with open(output_file, 'w') as f:
        json.dump(result, f, indent=2)
    
    print(f"\n🎯 Analysis Complete!")
    print(f"📄 JSON saved to: {output_file}")
    print(f"📊 Total rare regions: {len(all_rare_regions)}")
    return output_file


def find_multiline_code_in_chunk(code_snippet: str, chunk_content: str) -> dict:
    """
    Find multi-line code snippets in chunk using smart matching strategies.
    
    Strategies (in order):
    1. First-line exact match (use first line as anchor)
    2. Full exact match (for single-line or perfect copies)
    3. Normalized whitespace match (handles spacing differences)
    4. Keyword-based fuzzy match (fallback)
    """
    chunk_lines = chunk_content.split('\n')
    snippet_clean = code_snippet.strip()
    
    # Split snippet into lines
    snippet_lines = snippet_clean.split('\n')
    num_snippet_lines = len(snippet_lines)
    
    # STRATEGY 1: First-line anchor match (most reliable for multi-line)
    if num_snippet_lines > 0:
        first_line = snippet_lines[0].strip()
        
        for i, chunk_line in enumerate(chunk_lines):
            chunk_line_clean = chunk_line.strip()
            
            # Check if first line matches
            if first_line == chunk_line_clean or first_line in chunk_line_clean:
                return {
                    "line_in_chunk": i + 1,
                    "matched_line": chunk_line_clean,
                    "match_type": "first_line_exact",
                    "line_range": f"{i + 1}" if num_snippet_lines == 1 else f"{i + 1}-{i + num_snippet_lines}"
                }
    
    # STRATEGY 2: Full exact match (single line or perfect multi-line)
    for i, chunk_line in enumerate(chunk_lines):
        if snippet_clean in chunk_line.strip():
            return {
                "line_in_chunk": i + 1,
                "matched_line": chunk_line.strip(),
                "match_type": "full_exact",
                "line_range": str(i + 1)
            }
    
    # STRATEGY 3: Normalized whitespace match
    # Remove all whitespace and compare
    snippet_normalized = re.sub(r'\s+', '', snippet_clean.lower())
    
    # Try sliding window for multi-line matches
    for i in range(len(chunk_lines)):
        # Try windows of different sizes
        for window_size in range(1, min(num_snippet_lines + 3, len(chunk_lines) - i + 1)):
            window_lines = chunk_lines[i:i + window_size]
            window_text = '\n'.join(window_lines)
            window_normalized = re.sub(r'\s+', '', window_text.lower())
            
            # Check if normalized snippet is in normalized window
            if snippet_normalized in window_normalized:
                return {
                    "line_in_chunk": i + 1,
                    "matched_line": chunk_lines[i].strip(),
                    "match_type": "normalized_whitespace",
                    "line_range": f"{i + 1}" if window_size == 1 else f"{i + 1}-{i + window_size}"
                }
    
    # STRATEGY 4: Keyword-based fuzzy match (last resort)
    # Extract important keywords from first line
    first_line = snippet_lines[0] if snippet_lines else snippet_clean
    keywords = extract_keywords(first_line)
    
    if len(keywords) >= 2:
        best_match_line = None
        best_match_score = 0
        
        for i, chunk_line in enumerate(chunk_lines):
            line_keywords = set(re.findall(r'\w+', chunk_line.lower()))
            common_keywords = keywords & line_keywords
            
            # Calculate match score
            score = len(common_keywords) / len(keywords) if keywords else 0
            
            if score > best_match_score and score >= 0.5:  # 50% threshold
                best_match_score = score
                best_match_line = i
        
        if best_match_line is not None:
            return {
                "line_in_chunk": best_match_line + 1,
                "matched_line": chunk_lines[best_match_line].strip(),
                "match_type": f"fuzzy_keywords_{best_match_score:.2f}",
                "line_range": f"{best_match_line + 1}" if num_snippet_lines == 1 else f"{best_match_line + 1}-{best_match_line + num_snippet_lines}"
            }
    
    # No match found
    return None


def extract_keywords(code_line: str) -> set:
    """
    Extract meaningful keywords from code, filtering common noise.
    """
    # Verilog/SystemVerilog noise words
    noise = {
        'wire', 'reg', 'input', 'output', 'logic', 'assign', 'begin', 'end',
        'if', 'else', 'case', 'for', 'while', 'module', 'endmodule',
        'always', 'initial', 'posedge', 'negedge'
    }
    
    words = re.findall(r'\w+', code_line.lower())
    return set(w for w in words if w not in noise and len(w) > 1)


def process_gpt_response_with_location(response_text, chunk_id, chunk):
    """
    Process GPT JSON response and find code locations within chunk.
    """
    lines = response_text.strip()
    regions = []
    
    # Parse JSON
    try:
        if lines.startswith('[') and lines.endswith(']'):
            json_data = json.loads(lines)
            
            for region in json_data:
                code_snippet = region.get('code_snippet', '')
                
                # Find location using multi-strategy matching
                location_info = find_multiline_code_in_chunk(code_snippet, chunk['content'])
                
                # Build region entry
                enhanced_region = {
                    "chunk_id": chunk_id,
                    "rare_region_type": region.get('rare_region_type', 'UNKNOWN'),
                    "rare_region_code": code_snippet,
                    "activation_trigger": region.get('activation_trigger', 'Unknown trigger'),
                    "confidence_score": region.get('confidence_score', 0.5),
                    "description": region.get('description', 'No description provided')
                }
                
                # Add location info
                if location_info:
                    enhanced_region.update({
                        "line_in_chunk": location_info["line_in_chunk"],
                        "absolute_line": chunk['start_line'] + location_info["line_in_chunk"] - 1,
                        "matched_line_content": location_info["matched_line"],
                        "match_type": location_info["match_type"],
                        "line_range": location_info["line_range"]
                    })
                    
                    preview = code_snippet.replace('\n', ' ')[:50]
                    print(f"   📍 Found at line {location_info['line_range']} ({location_info['match_type']})")
                else:
                    enhanced_region.update({
                        "line_in_chunk": None,
                        "absolute_line": None,
                        "matched_line_content": None,
                        "match_type": "not_found",
                        "line_range": None
                    })
                    preview = code_snippet.replace('\n', ' ')[:50]
                    print(f"   ⚠️ Could not locate: {preview}...")
                
                regions.append(enhanced_region)
            
            return regions
            
    except json.JSONDecodeError as e:
        print(f"   ❌ JSON parsing failed: {e}")
    
    return regions


def view_results(json_file):
    """View results in readable format"""
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    print("📊 RARE REGIONS ANALYSIS RESULTS")
    print("=" * 50)
    print(f"📁 Source: {data['file_metadata']['source_file']}")
    print(f"📈 Total Chunks: {data['file_metadata']['total_chunks_analyzed']}")
    print(f"🎯 Chunks with Regions: {data['file_metadata']['chunks_with_rare_regions']}")
    print(f"🔍 Total Rare Regions: {data['file_metadata']['total_rare_regions']}")
    
    print("\n🧩 RARE REGIONS BY CHUNK:")
    print("-" * 50)
    
    current_chunk = None
    for region in data['rare_regions']:
        if region['chunk_id'] != current_chunk:
            current_chunk = region['chunk_id']
            print(f"\n📦 {current_chunk}:")
        
        code_preview = region['rare_region_code'].replace('\n', ' ')[:50] + '...'
        line_info = region.get('line_range', 'N/A')
        
        print(f"  • {region['rare_region_type']}")
        print(f"    📍 Lines: {line_info} | Match: {region.get('match_type', 'N/A')}")
        print(f"    Code: {code_preview}")


def json_to_csv(json_file):
    """Convert JSON to CSV"""
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data['rare_regions'])
    csv_file = json_file.replace('.json', '.csv')
    df.to_csv(csv_file, index=False)
    print(f"✅ CSV saved to: {csv_file}")
    return csv_file


# Ready to use
print("🚀 SIMPLIFIED RARE REGION ANALYSIS READY!")
print("=" * 50)
print("\n📝 Usage:")
print("  interactive_analysis('/kaggle/working/filename_chunks.json')")
print("  view_results('/kaggle/working/filename_rare_regions.json')")
print("\n✨ Features:")
print("  • Clean, simple prompt - just asks for EXACT VERBATIM code")
print("  • 4-strategy matching: exact → first-line → normalized → fuzzy")
print("  • Handles single-line and multi-line code blocks")
print("  • Reports line ranges for multi-line matches")
print("\n🎯 Ready to start!")

## Cell 13: Smart Pipeline Wrapper

In [ ]:
# Cell 13: Smart Pipeline Wrapper
# Uses existing functions with flexibility for different starting points

def analyze_file_complete(file_path: str, remove_comments: bool = True):
    """
    Complete pipeline: file → chunks → rare regions
    
    Args:
        file_path: Path to .sv or .v file
        remove_comments: Whether to remove comments (default: True)
    """
    print("🚀 COMPLETE PIPELINE: File → Chunks → Rare Regions")
    print("=" * 55)
    
    # Step 1: Generate chunks
    print("\n📦 STEP 1: Generating chunks...")
    generator = SequentialChunkGenerator()
    generator.remove_comments_enabled = remove_comments
    
    chunks_data = generator.generate_chunks(file_path)
    
    # Save chunks
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    chunks_file = f"/kaggle/working/{base_name}_chunks.json"
    save_chunks_to_file(chunks_data, chunks_file)
    
    # Step 2: Analyze rare regions
    print(f"\n🔍 STEP 2: Analyzing rare regions...")
    rare_regions_file = interactive_analysis(chunks_file)
    
    print(f"\n✅ COMPLETE! Final output: {rare_regions_file}")
    return rare_regions_file

def analyze_from_chunks(chunks_file: str):
    """
    Start from existing chunks file → rare regions only
    
    Args:
        chunks_file: Path to existing filename_chunks.json
    """
    print("🔍 RARE REGION ANALYSIS: Starting from existing chunks")
    print("=" * 55)
    
    if not os.path.exists(chunks_file):
        print(f"❌ Chunks file not found: {chunks_file}")
        return None
    
    print(f"📦 Using existing chunks: {os.path.basename(chunks_file)}")
    rare_regions_file = interactive_analysis(chunks_file)
    
    print(f"\n✅ COMPLETE! Rare regions: {rare_regions_file}")
    return rare_regions_file

def quick_chunks_only(file_path: str, remove_comments: bool = True):
    """
    Generate chunks only (no rare region analysis)
    
    Args:
        file_path: Path to .sv or .v file
        remove_comments: Whether to remove comments (default: True)
    """
    print("📦 CHUNKS ONLY: Generating chunks without rare region analysis")
    print("=" * 60)
    
    generator = SequentialChunkGenerator()
    generator.remove_comments_enabled = remove_comments
    
    chunks_data = generator.generate_chunks(file_path)
    
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    chunks_file = f"/kaggle/working/{base_name}_chunks.json"
    save_chunks_to_file(chunks_data, chunks_file)
    
    print(f"\n✅ Chunks saved: {chunks_file}")
    print("💡 Next step: analyze_from_chunks('{chunks_file}') for rare regions")
    return chunks_file

def check_existing_files(file_path: str):
    """
    Check what files already exist and suggest next steps
    
    Args:
        file_path: Path to .sv or .v file
    """
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    chunks_file = f"/kaggle/working/{base_name}_chunks.json"
    rare_regions_file = f"/kaggle/working/{base_name}_rare_regions.json"
    
    print("🔍 CHECKING EXISTING FILES")
    print("=" * 30)
    print(f"📁 Base file: {os.path.basename(file_path)}")
    
    chunks_exists = os.path.exists(chunks_file)
    rare_regions_exists = os.path.exists(rare_regions_file)
    
    print(f"📦 Chunks file: {os.path.basename(chunks_file)} {'✅ EXISTS' if chunks_exists else '❌ NOT FOUND'}")
    print(f"🔍 Rare regions: {os.path.basename(rare_regions_file)} {'✅ EXISTS' if rare_regions_exists else '❌ NOT FOUND'}")
    
    print(f"\n💡 RECOMMENDED NEXT STEPS:")
    
    if rare_regions_exists:
        print("   🎉 Analysis already complete!")
        print(f"   📋 View results: view_results('{rare_regions_file}')")
    elif chunks_exists:
        print("   📦 Chunks ready - run rare region analysis:")
        print(f"   🔍 analyze_from_chunks('{chunks_file}')")
    else:
        print("   🚀 Start fresh - complete pipeline:")
        print(f"   📝 analyze_file_complete('{file_path}')")
        print("   OR")
        print("   📦 Chunks only first:")
        print(f"   🔧 quick_chunks_only('{file_path}')")
    
    return {
        "chunks_exists": chunks_exists,
        "rare_regions_exists": rare_regions_exists,
        "chunks_file": chunks_file,
        "rare_regions_file": rare_regions_file
    }

# Simple wrapper functions for common use cases
def analyze_file(file_path: str):
    """Simple complete analysis - most common usage"""
    return analyze_file_complete(file_path)

def analyze_chunks(chunks_file: str):
    """Simple rare region analysis from chunks"""
    return analyze_from_chunks(chunks_file)

# Smart auto-detection function
def smart_analyze(file_path: str):
    """
    Smart analysis - automatically detects what to do based on existing files
    
    Args:
        file_path: Path to .sv or .v file
    """
    status = check_existing_files(file_path)
    
    if status["rare_regions_exists"]:
        print(f"\n🎉 Analysis already complete!")
        view_results(status["rare_regions_file"])
        return status["rare_regions_file"]
    elif status["chunks_exists"]:
        print(f"\n📦 Found existing chunks, starting rare region analysis...")
        return analyze_from_chunks(status["chunks_file"])
    else:
        print(f"\n🚀 Starting complete analysis from scratch...")
        return analyze_file_complete(file_path)

# Usage examples and help
print("🎯 FLEXIBLE PIPELINE READY!")
print("=" * 35)
print("")
print("🔧 USAGE OPTIONS:")
print("")
print("1️⃣ COMPLETE ANALYSIS (file → chunks → rare regions):")
print("   analyze_file('/path/to/file.sv')")
print("")
print("2️⃣ RARE REGIONS ONLY (from existing chunks):")
print("   analyze_chunks('/kaggle/working/filename_chunks.json')")
print("")
print("3️⃣ CHUNKS ONLY (no rare region analysis):")
print("   quick_chunks_only('/path/to/file.sv')")
print("")
print("4️⃣ CHECK STATUS (see what files exist):")
print("   check_existing_files('/path/to/file.sv')")
print("")
print("5️⃣ SMART AUTO (detects what to do automatically):")
print("   smart_analyze('/path/to/file.sv')")
print("")
print("📋 VIEW RESULTS:")
print("   view_results('/kaggle/working/filename_rare_regions.json')")
print("")
print("💡 MOST COMMON:")
print("   🚀 smart_analyze('/kaggle/input/riscv-ht/RISCV.sv')")

In [ ]:
analyze_chunks('/kaggle/working/RISCV_chunks.json')

# HT_TYPE_ANALYSIS

In [ ]:
# Enhanced Hardware Trojan Type Analyzer with Security Research Context
# Updated prompt emphasizing academic security research purpose

import json
import re
from datetime import datetime
from typing import Dict, List, Optional, Tuple

class HardwareTrojanTypeAnalyzer:
    """
    Enhanced analyzer for determining optimal HT types for rare regions.
    Designed for hardware security research and vulnerability analysis.
    """
    
    def __init__(self):
        self.ht_types = {
            "HT1_Crypto": {
                "name": "Cryptographic Attacks",
                "description": "Targets encryption, key manipulation, crypto bypass",
                "targets": ["encryption modules", "key storage", "crypto operations", "random number generators"]
            },
            "HT1_Privilege": {
                "name": "Privilege Escalation", 
                "description": "Bypasses access controls, escalates privileges",
                "targets": ["control logic", "permission checks", "state machines", "access control"]
            },
            "HT2_Leakage": {
                "name": "Information Leakage",
                "description": "Creates covert channels for data exfiltration", 
                "targets": ["memory interfaces", "register files", "data paths", "reset logic"]
            },
            "HT3_Performance": {
                "name": "Performance/DoS Attacks",
                "description": "Causes system failure, lockup, or performance degradation",
                "targets": ["clock systems", "reset logic", "critical paths", "resource management"]
            }
        }
    
    def interactive_ht_analysis(self, rare_regions_file: str, chunks_file: str) -> str:
        """
        Interactive analysis to identify best HT type for each rare region.
        Enhanced with security research context and vulnerability study focus.
        """
        # Load data
        with open(rare_regions_file, 'r') as f:
            rare_regions_data = json.load(f)
        
        with open(chunks_file, 'r') as f:
            chunks_data = json.load(f)
        
        all_ht_analysis = []
        total_regions = len(rare_regions_data['rare_regions'])
        
        print(f"🔬 HARDWARE SECURITY RESEARCH: HT VULNERABILITY ANALYSIS")
        print(f"Analyzing {total_regions} rare regions for security vulnerabilities...")
        print("=" * 80)
        
        for idx, region in enumerate(rare_regions_data['rare_regions']):
            region_id = f"{region['chunk_id']}_{region['rare_region_type']}"
            
            # Enhanced code context with marked rare region
            code_context, marked_lines = self.find_enhanced_code_context(region, chunks_data)
            
            # Display analysis prompt with security research context
            self.display_security_research_prompt(region, region_id, code_context, marked_lines, idx + 1, total_regions)
            
            # Collect LLM response
            print(f"\n📋 Paste LLM response for {region_id} (type 'DONE' on new line when finished):")
            print(f"💡 If no suitable HT type exists, respond with: [] ")
            
            gpt_response = []
            while True:
                line = input()
                if line.strip() == 'DONE':
                    break
                gpt_response.append(line)
            
            # Process response
            response_text = '\n'.join(gpt_response)
            ht_analysis = self.process_ht_response(response_text, region, region_id, code_context)
            
            if ht_analysis:
                all_ht_analysis.append(ht_analysis)
                print(f"✅ Processed HT analysis for {region_id}")
                if ht_analysis.get('no_suitable_ht', False):
                    print("   📝 Note: No suitable HT type identified for this region")
            else:
                print(f"⚠️ Failed to process response for {region_id}")
            
            print()  # Add spacing between regions
        
        # Create output file
        output_file = rare_regions_file.replace('_rare_regions.json', '_ht_analysis.json')
        
        # Get source file from metadata with fallback
        source_file = "unknown"
        if 'file_metadata' in rare_regions_data:
            source_file = rare_regions_data['file_metadata'].get('source_file', 
                         rare_regions_data['file_metadata'].get('filename', 'unknown'))
        
        result = {
            "trojan_analysis": {
                "metadata": {
                    "source_file": source_file,
                    "total_rare_regions": total_regions,
                    "regions_analyzed": len(all_ht_analysis),
                    "exploitable_regions": len([r for r in all_ht_analysis if not r.get('no_suitable_ht', False)]),
                    "analysis_timestamp": datetime.now().isoformat(),
                    "research_purpose": "Hardware security vulnerability analysis and trojan susceptibility study"
                },
                "region_analysis": all_ht_analysis
            }
        }
        
        with open(output_file, 'w') as f:
            json.dump(result, f, indent=2)
        
        # Summary
        exploitable = len([r for r in all_ht_analysis if not r.get('no_suitable_ht', False)])
        print(f"\n🎯 HT VULNERABILITY ANALYSIS COMPLETE!")
        print(f"📄 Results saved to: {output_file}")
        print(f"📊 Total regions analyzed: {len(all_ht_analysis)}")
        print(f"🎯 Exploitable regions: {exploitable}/{total_regions}")
        print(f"🛡️ Secure regions: {total_regions - exploitable}/{total_regions}")
        
        return output_file
    
    def display_security_research_prompt(self, region: Dict, region_id: str, code_context: List[str], 
                                        marked_lines: List[int], current: int, total: int):
        """
        Display comprehensive analysis prompt with security research context.
        """
        
        print(f"\n🔬 SECURITY ANALYSIS {current}/{total}: {region['rare_region_type']} in {region['chunk_id']}")
        print("=" * 80)
        
        # Research context
        print(f"🎓 RESEARCH PURPOSE:")
        print(f"   This analysis is for HARDWARE SECURITY RESEARCH to study vulnerable rare regions.")
        print(f"   Goal: Identify which Hardware Trojan types could exploit this rare execution path,")
        print(f"   remain hidden during testing, and cause significant damage in deployed systems.")
        print(f"   This research helps improve hardware security defenses and detection methods.")
        print()
        
        # Region details
        print(f"📍 RARE REGION DETAILS:")
        print(f"   Code: {region.get('rare_region_code', region.get('code_snippet', 'N/A'))[:60]}...")
        print(f"   Trigger: {region['activation_trigger']}")
        print(f"   Confidence: {region['confidence_score']}")
        print(f"   Description: {region['description']}")
        
        # Code context with marked rare region
        print(f"\n🔍 CODE CONTEXT (rare region marked with <-- RARE REGION):")
        print("─" * 60)
        for line_num, line in enumerate(code_context, 1):
            marker = ">>>" if line_num - 1 in marked_lines else "   "
            print(f"{marker} {line_num:2d}: {line}")
        print("─" * 60)
        
        # Security research task
        print(f"\n🎯 SECURITY RESEARCH TASK:")
        print(f"Analyze what kind of Hardware Trojan (HT) can:")
        print(f"  1. STAY HIDDEN in this rare region during normal operation and testing")
        print(f"  2. CAUSE HEAVY DAMAGE when activated in deployed systems")
        print(f"  3. REMAIN STEALTHY during functional verification and security testing")
        print(f"  4. Have a REALISTIC PAYLOAD LOGIC that exploits this vulnerability")
        print()
        print(f"This vulnerability analysis helps us:")
        print(f"  • Understand attack surface of rare execution regions")
        print(f"  • Develop better hardware security testing methodologies")
        print(f"  • Design trojan-resistant hardware architectures")
        print(f"  • Improve detection techniques for malicious hardware modifications")
        
        print(f"\n🔬 EVALUATE THESE HT TYPES FOR THIS VULNERABLE REGION:")
        for ht_type, info in self.ht_types.items():
            print(f"  • {ht_type}: {info['description']}")
        
        print(f"\n📋 REQUIRED ANALYSIS (for security research):")
        print(f"For this vulnerable rare region, determine:")
        print(f"  1. BEST HT TYPE - Which trojan category fits this vulnerability")
        print(f"  2. DETAILED RATIONALE - WHY this rare region is susceptible to this HT type")
        print(f"  3. PAYLOAD LOGIC - Specific attack mechanism that exploits this rare execution path")
        print(f"  4. STEALTH ASSESSMENT - How the trojan remains undetected during testing")
        print(f"     (Low/Medium/High/Very High)")
        print(f"  5. DAMAGE POTENTIAL - Impact if trojan activates in production")
        print(f"     (Low/Medium/High/Critical)")
        print(f"  6. DETECTION DIFFICULTY - How hard to find using current methods")
        print(f"     (Easy/Medium/Hard/Very Hard/Extremely Hard)")
        
        print(f"\n💡 IMPORTANT FOR RESEARCH:")
        print(f"  • Focus on REALISTIC attack scenarios that could occur in supply chain")
        print(f"  • Consider how rare execution makes trojan harder to detect")
        print(f"  • Explain specific payload logic that exploits this vulnerability")
        print(f"  • If NO suitable HT type exists for this region, respond with: []")
        
        print(f"\n🎯 REQUIRED JSON OUTPUT FORMAT:")
        print('{')
        print('  "best_ht_type": "HT2_Leakage",')
        print('  "rationale": "This rare reset sequence has privileged access to all registers. The infrequent execution makes trojan activation nearly impossible to observe during testing, while still allowing data exfiltration during rare reset events in production.",')
        print('  "attack_method": "Insert additional logic in reset path to sample register contents during reset and transmit them via unused GPIO pins using a timing-based covert channel. Payload activates only during rare reset conditions, making it undetectable during normal testing.",')
        print('  "stealth_level": "Very High",')
        print('  "damage_potential": "Critical",')
        print('  "detection_difficulty": "Extremely Hard"')
        print('}')
        print("\nFor non-exploitable regions: []")
        
        print(f"\n📊 Progress: {current}/{total} regions analyzed")
        print("=" * 80)
    
    def find_enhanced_code_context(self, region: Dict, chunks_data: Dict) -> Tuple[List[str], List[int]]:
        """
        Find enhanced code context with marked rare region lines.
        Returns (context_lines, marked_line_indices)
        """
        chunk_id = region['chunk_id']
        code_snippet = region.get('rare_region_code', region.get('code_snippet', '')).strip()
        
        if chunk_id not in chunks_data['chunks']:
            return [code_snippet], [0]
        
        chunk_content = chunks_data['chunks'][chunk_id]['content']
        lines = chunk_content.split('\n')
        
        # Find target line(s) containing the code snippet
        target_lines = []
        for i, line in enumerate(lines):
            if self.snippet_matches_line(code_snippet, line):
                target_lines.append(i)
        
        if not target_lines:
            target_lines = self.fuzzy_find_snippet(code_snippet, lines)
        
        if not target_lines:
            return [f"// RARE REGION: {code_snippet}"], [0]
        
        # Extract context around target lines
        context_start = max(0, min(target_lines) - 7)
        context_end = min(len(lines), max(target_lines) + 8)
        
        context_lines = []
        marked_lines = []
        
        for i in range(context_start, context_end):
            if i < len(lines):
                line = lines[i].rstrip()
                if i in target_lines:
                    marked_line = f"{line}  // <-- RARE REGION"
                    context_lines.append(marked_line)
                    marked_lines.append(len(context_lines) - 1)
                else:
                    context_lines.append(line)
        
        return context_lines, marked_lines
    
    def snippet_matches_line(self, snippet: str, line: str) -> bool:
        """Check if snippet matches line"""
        clean_snippet = re.sub(r'\s+', ' ', snippet.strip())
        clean_line = re.sub(r'\s+', ' ', line.strip())
        
        if clean_snippet in clean_line:
            return True
        
        snippet_keywords = re.findall(r'\w+', clean_snippet.lower())
        line_keywords = re.findall(r'\w+', clean_line.lower())
        
        if len(snippet_keywords) >= 2:
            matches = sum(1 for kw in snippet_keywords if kw in line_keywords)
            return matches >= len(snippet_keywords) * 0.7
        
        return False
    
    def fuzzy_find_snippet(self, snippet: str, lines: List[str]) -> List[int]:
        """Fuzzy search for snippet in lines"""
        snippet_words = set(re.findall(r'\w+', snippet.lower()))
        if len(snippet_words) < 2:
            return []
        
        best_matches = []
        for i, line in enumerate(lines):
            line_words = set(re.findall(r'\w+', line.lower()))
            common = snippet_words.intersection(line_words)
            
            if len(common) >= len(snippet_words) * 0.6:
                best_matches.append(i)
        
        return best_matches[:3]
    
    def process_ht_response(self, response_text: str, original_region: Dict, 
                          region_id: str, code_context: List[str]) -> Optional[Dict]:
        """Process LLM response into structured analysis"""
        
        # Check for empty/no suitable HT response
        if response_text.strip() in ['[]', '', 'null']:
            return {
                "region_id": region_id,
                "chunk_id": original_region['chunk_id'],
                "original_rare_region": {
                    "rare_region_type": original_region['rare_region_type'],
                    "code_snippet": original_region.get('rare_region_code', original_region.get('code_snippet', 'N/A')),
                    "activation_trigger": original_region['activation_trigger'],
                    "confidence_score": original_region['confidence_score'],
                    "description": original_region['description']
                },
                "code_context": code_context,
                "no_suitable_ht": True,
                "analysis_note": "No suitable hardware trojan type identified for this region"
            }
        
        try:
            # Try to parse JSON response
            if '{' in response_text and '}' in response_text:
                start = response_text.find('{')
                end = response_text.rfind('}') + 1
                json_str = response_text[start:end]
                ht_data = json.loads(json_str)
                
                # Validate required fields
                required_fields = ['best_ht_type', 'rationale', 'attack_method', 
                                 'stealth_level', 'damage_potential', 'detection_difficulty']
                
                if all(field in ht_data for field in required_fields):
                    return {
                        "region_id": region_id,
                        "chunk_id": original_region['chunk_id'],
                        "original_rare_region": {
                            "rare_region_type": original_region['rare_region_type'],
                            "code_snippet": original_region.get('rare_region_code', original_region.get('code_snippet', 'N/A')),
                            "activation_trigger": original_region['activation_trigger'],
                            "confidence_score": original_region['confidence_score'],
                            "description": original_region['description']
                        },
                        "code_context": code_context,
                        "best_ht_match": {
                            "trojan_type": ht_data['best_ht_type'],
                            "rationale": ht_data['rationale'],
                            "attack_method": ht_data['attack_method'],
                            "stealth_level": ht_data['stealth_level'],
                            "damage_potential": ht_data['damage_potential'],
                            "detection_difficulty": ht_data['detection_difficulty']
                        },
                        "no_suitable_ht": False
                    }
        except json.JSONDecodeError:
            pass
        
        print(f"⚠️ Warning: Could not parse response for {region_id}")
        return None

    def view_ht_analysis_results(self, ht_analysis_file: str):
        """View HT analysis results"""
        with open(ht_analysis_file, 'r') as f:
            data = json.load(f)
        
        metadata = data['trojan_analysis']['metadata']
        regions = data['trojan_analysis']['region_analysis']
        
        print("🔬 HARDWARE TROJAN VULNERABILITY ANALYSIS RESULTS")
        print("=" * 70)
        print(f"📁 Source: {metadata['source_file']}")
        print(f"📈 Total Regions: {metadata['total_rare_regions']}")
        print(f"🔍 Analyzed: {metadata['regions_analyzed']}")
        print(f"🎯 Exploitable: {metadata['exploitable_regions']}")
        print(f"🛡️ Secure: {metadata['total_rare_regions'] - metadata['exploitable_regions']}")
        print(f"📅 Analysis Time: {metadata['analysis_timestamp']}")
        
        print(f"\n🎯 VULNERABLE REGIONS (Exploitable by HT):")
        print("-" * 70)
        
        for analysis in regions:
            if not analysis.get('no_suitable_ht', False):
                best_ht = analysis['best_ht_match']
                print(f"\n📦 {analysis['region_id']}:")
                print(f"  🎯 HT Type: {best_ht['trojan_type']}")
                print(f"  💀 Damage: {best_ht['damage_potential']}")
                print(f"  👻 Stealth: {best_ht['stealth_level']}")
                print(f"  🔍 Detection: {best_ht['detection_difficulty']}")
                print(f"  📝 Attack: {best_ht['attack_method'][:80]}...")

    def summarize_ht_distribution(self, ht_analysis_file: str):
        """Summarize HT type distribution"""
        with open(ht_analysis_file, 'r') as f:
            data = json.load(f)
        
        regions = data['trojan_analysis']['region_analysis']
        exploitable = [r for r in regions if not r.get('no_suitable_ht', False)]
        
        print("📊 HARDWARE TROJAN VULNERABILITY DISTRIBUTION")
        print("=" * 50)
        
        ht_counts = {}
        stealth_counts = {}
        damage_counts = {}
        
        for analysis in exploitable:
            ht_type = analysis['best_ht_match']['trojan_type']
            stealth = analysis['best_ht_match']['stealth_level']
            damage = analysis['best_ht_match']['damage_potential']
            
            ht_counts[ht_type] = ht_counts.get(ht_type, 0) + 1
            stealth_counts[stealth] = stealth_counts.get(stealth, 0) + 1
            damage_counts[damage] = damage_counts.get(damage, 0) + 1
        
        print(f"🎯 HT TYPE BREAKDOWN:")
        for ht_type, count in sorted(ht_counts.items()):
            percentage = (count / len(exploitable)) * 100 if exploitable else 0
            print(f"  {ht_type}: {count} regions ({percentage:.1f}%)")
        
        print(f"\n👻 STEALTH LEVEL:")
        for level, count in stealth_counts.items():
            if count > 0:
                percentage = (count / len(exploitable)) * 100
                print(f"  {level}: {count} ({percentage:.1f}%)")
        
        print(f"\n💀 DAMAGE POTENTIAL:")
        for level, count in damage_counts.items():
            if count > 0:
                percentage = (count / len(exploitable)) * 100
                print(f"  {level}: {count} ({percentage:.1f}%)")

# Create analyzer instance
ht_analyzer = HardwareTrojanTypeAnalyzer()

print("🔬 HARDWARE SECURITY RESEARCH: HT VULNERABILITY ANALYZER")
print("=" * 60)
print("📝 Usage:")
print("1. ht_analyzer.interactive_ht_analysis('file_rare_regions.json', 'file_chunks.json')")
print("2. ht_analyzer.view_ht_analysis_results('file_ht_analysis.json')")  
print("3. ht_analyzer.summarize_ht_distribution('file_ht_analysis.json')")
print("\n🎓 Research Purpose:")
print("  Study vulnerable rare regions to improve hardware security defenses")
print("  Analyze trojan susceptibility for better detection methodologies")
print("  Understand attack surface of infrequently-executed code paths")
print("\n🎯 Ready for security vulnerability analysis!")

In [ ]:
ht_analysis_file = ht_analyzer.interactive_ht_analysis(
    '/kaggle/working/RISCV_rare_regions.json', 
    '/kaggle/working/RISCV_chunks.json'
)

# HT IMPLEMENTATION

In [ ]:
import json
import re
import os
from datetime import datetime
from typing import Dict, List, Tuple, Optional

class ImprovedHTImplementationPipeline:
    """
    HT implementation with signal validation and scope filtering.
    Filters available signals based on declaration line vs context line.
    """
    
    def __init__(self):
        self.implementations = []
        self.signals_data = None
    
    def load_data(self, ht_analysis_file: str, rare_regions_file: str, 
                  chunks_file: str, signals_file: str = None):
        """Load all required JSON files including signals."""
        with open(ht_analysis_file, 'r') as f:
            self.ht_data = json.load(f)
        
        with open(rare_regions_file, 'r') as f:
            self.rare_regions_data = json.load(f)
        
        with open(chunks_file, 'r') as f:
            self.chunks_data = json.load(f)
        
        if signals_file and os.path.exists(signals_file):
            with open(signals_file, 'r') as f:
                self.signals_data = json.load(f)
            print(f"Loaded signals for {len(self.signals_data.get('module_signals', {}))} modules")
        
        print(f"Loaded {len(self.ht_data['trojan_analysis']['region_analysis'])} HT analyses")
        print(f"Loaded {len(self.rare_regions_data['rare_regions'])} rare regions")
        print(f"Loaded {len(self.chunks_data['chunks'])} chunks")
    
    def get_module_id(self, chunk_id: str) -> str:
        """Extract module ID from chunk ID."""
        parts = chunk_id.split('_')
        if len(parts) >= 2:
            return f"{parts[0]}_{parts[1]}"
        return chunk_id
    
    def filter_available_signals(self, signals_info: Dict, context_start_line: int) -> Dict:
        """
        Filter signals based on declaration line vs context start line.
        Only include signals declared BEFORE context starts.
        """
        if not signals_info:
            return None
        
        filtered = {
            'ports': [],
            'module_signals': [],
            'parameters': [],
            'enums': [],
            'state_signals': []
        }
        
        # Ports - always available (declared in module header)
        filtered['ports'] = signals_info.get('ports', [])
        
        # Parameters - always available (declared before logic)
        filtered['parameters'] = signals_info.get('parameters', [])
        
        # Module signals - filter by declaration line
        for sig in signals_info.get('module_signals', []):
            if sig.get('declared_at_line', 0) < context_start_line:
                filtered['module_signals'].append(sig)
        
        # Enums - filter by declaration line
        for enum in signals_info.get('enums', []):
            if enum.get('declared_at_line', 0) < context_start_line:
                filtered['enums'].append(enum)
        
        # State signals - filter by declaration line
        for sig in signals_info.get('state_signals', []):
            if sig.get('declared_at_line', 0) < context_start_line:
                filtered['state_signals'].append(sig)
        
        return filtered
    
    def get_available_signals_for_context(self, chunk_id: str, context_start_line: int) -> Dict:
        """Get signals available at context start line."""
        if not self.signals_data:
            return None
        
        module_id = self.get_module_id(chunk_id)
        signals_info = self.signals_data.get('module_signals', {}).get(module_id)
        
        if not signals_info:
            return None
        
        return self.filter_available_signals(signals_info, context_start_line)
    
    def format_available_signals_display(self, signals_info: Dict) -> str:
        """Format filtered signals info for GPT prompt."""
        if not signals_info:
            return "WARNING: No signal information available - be extra careful with signal names!"
        
        lines = []
        lines.append("=== AVAILABLE SIGNALS (declared before context) ===")
        lines.append("You can ONLY use these existing signals:\n")
        
        if signals_info.get('ports'):
            lines.append("MODULE PORTS:")
            for port in signals_info['ports']:
                width = f" {port['width']}" if port.get('width') else ""
                lines.append(f"  - {port['name']}{width}: {port['direction']} {port['type']}")
            lines.append("")
        
        if signals_info.get('module_signals'):
            lines.append("MODULE-LEVEL SIGNALS:")
            for sig in signals_info['module_signals']:
                width = f" {sig['width']}" if sig.get('width') else ""
                lines.append(f"  - {sig['name']}{width}: {sig['type']} (declared line {sig['declared_at_line']})")
            lines.append("")
        
        if signals_info.get('parameters'):
            lines.append("PARAMETERS:")
            for param in signals_info['parameters']:
                value = f" = {param['value']}" if param.get('value') else ""
                lines.append(f"  - {param['name']}: {param['type']}{value}")
            lines.append("")
        
        if signals_info.get('enums'):
            lines.append("ENUMS/STATES:")
            for enum in signals_info['enums']:
                values = ', '.join(enum['values'])
                lines.append(f"  - {enum['name']}: {enum['type']}")
                lines.append(f"    Values: {values}")
            lines.append("")
        
        if signals_info.get('state_signals'):
            lines.append("STATE MACHINE SIGNALS:")
            for sig in signals_info['state_signals']:
                lines.append(f"  - {sig['name']}: {sig['type']}")
            lines.append("")
        
        return '\n'.join(lines)
    
    def find_rare_region_in_chunk(self, chunk_id: str, rare_region_code: str) -> Optional[Tuple[int, int]]:
        """Find exact line range of rare region code in chunk."""
        if chunk_id not in self.chunks_data['chunks']:
            return None
        
        chunk_content = self.chunks_data['chunks'][chunk_id]['content']
        chunk_lines = chunk_content.split('\n')
        rare_lines = [line.strip() for line in rare_region_code.split('\n') if line.strip()]
        
        if not rare_lines:
            return None
        
        for i in range(len(chunk_lines)):
            if rare_lines[0] in chunk_lines[i] or chunk_lines[i].strip() == rare_lines[0]:
                match = True
                for j, rare_line in enumerate(rare_lines):
                    if i + j >= len(chunk_lines):
                        match = False
                        break
                    
                    chunk_line_clean = chunk_lines[i + j].strip()
                    rare_line_clean = rare_line.strip()
                    
                    if rare_line_clean not in chunk_line_clean and chunk_line_clean != rare_line_clean:
                        if re.sub(r'\s+', '', rare_line_clean) not in re.sub(r'\s+', '', chunk_line_clean):
                            match = False
                            break
                
                if match:
                    return (i, i + len(rare_lines) - 1)
        
        return None
    
    def extract_context_block(self, chunk_id: str, rare_region_start: int, rare_region_end: int, 
                             context_lines: int = 3) -> Dict:
        """Extract context block around rare region with line numbers."""
        chunk = self.chunks_data['chunks'][chunk_id]
        chunk_lines = chunk['content'].split('\n')
        chunk_start_line = chunk['start_line']
        
        context_start = max(0, rare_region_start - context_lines)
        context_end = min(len(chunk_lines) - 1, rare_region_end + context_lines)
        
        context_block = []
        for i in range(context_start, context_end + 1):
            line_data = {
                "line_in_chunk": i,
                "absolute_line": chunk_start_line + i,
                "content": chunk_lines[i],
                "is_rare_region": rare_region_start <= i <= rare_region_end
            }
            context_block.append(line_data)
        
        return {
            "chunk_id": chunk_id,
            "context_start_line": context_start,
            "context_end_line": context_end,
            "rare_region_start_line": rare_region_start,
            "rare_region_end_line": rare_region_end,
            "absolute_start_line": chunk_start_line + context_start,
            "absolute_end_line": chunk_start_line + context_end,
            "context_block": context_block
        }
    
    def generate_improved_gpt_prompt(self, region_analysis: Dict) -> Tuple[str, Dict]:
        """Generate improved GPT prompt with filtered signal list."""
        rare_region = None
        for rr in self.rare_regions_data['rare_regions']:
            if (rr['chunk_id'] == region_analysis['chunk_id'] and 
                rr.get('rare_region_type') == region_analysis['original_rare_region'].get('rare_region_type')):
                rare_region = rr
                break
        
        if not rare_region:
            return None, None
        
        rare_region_code = rare_region.get('rare_region_code', rare_region.get('code_snippet', ''))
        location = self.find_rare_region_in_chunk(region_analysis['chunk_id'], rare_region_code)
        
        if not location:
            print(f"Warning: Could not locate rare region in {region_analysis['chunk_id']}")
            return None, None
        
        context_data = self.extract_context_block(
            region_analysis['chunk_id'], 
            location[0], 
            location[1],
            context_lines=3
        )
        
        # Get filtered signals (only those declared before context)
        available_signals = self.get_available_signals_for_context(
            region_analysis['chunk_id'],
            context_data['absolute_start_line']
        )
        signals_display = self.format_available_signals_display(available_signals)
        
        # Build context display
        context_display = []
        for line_data in context_data['context_block']:
            marker = ">>> RARE REGION >>>" if line_data['is_rare_region'] else "   "
            context_display.append(
                f"{marker} Line {line_data['absolute_line']:4d}: {line_data['content']}"
            )
        
        ht_match = region_analysis['best_ht_match']
        rare_info = region_analysis['original_rare_region']
        
        prompt = f"""HARDWARE TROJAN IMPLEMENTATION FOR SECURITY RESEARCH

=== TROJAN SPECIFICATION ===
Type: {ht_match['trojan_type']}
Stealth: {ht_match['stealth_level']}
Damage: {ht_match['damage_potential']}
Detection Difficulty: {ht_match['detection_difficulty']}

Rationale: {ht_match['rationale']}

Attack Method: {ht_match['attack_method']}

=== RARE REGION INFO ===
Type: {rare_info['rare_region_type']}
Trigger: {rare_info['activation_trigger']}
Confidence: {rare_info['confidence_score']}
Description: {rare_info['description']}

{signals_display}

=== TROJAN SIGNAL RULES ===
1. You can ONLY use signals from the available signals list above
2. For NEW signals, MUST use "trojan_" prefix
3. NO undefined references allowed
4. Example new signals:
   ✅ reg trojan_leak_buffer [31:0];
   ✅ logic trojan_trigger;
   ✅ reg [7:0] trojan_shift_reg;
   ❌ reg leak_buffer;  // Missing trojan_ prefix!
   ❌ reg shift_reg;    // Conflict with existing signal!

5. ALL new trojan signals must be:
   - Prefixed with "trojan_"
   - Listed in new_trojan_signals array
   - Declared in signal_declarations array

=== CODE CONTEXT (with line numbers) ===
Chunk: {context_data['chunk_id']}
Context Range: Lines {context_data['absolute_start_line']}-{context_data['absolute_end_line']}
Rare Region: Lines {context_data['absolute_start_line'] + (context_data['rare_region_start_line'] - context_data['context_start_line'])}-{context_data['absolute_start_line'] + (context_data['rare_region_end_line'] - context_data['context_start_line'])}

{chr(10).join(context_display)}

=== TASK ===
Implement the hardware trojan by modifying the COMPLETE context block shown above.

REQUIREMENTS:
1. Return the ENTIRE context block (all lines from {context_data['absolute_start_line']} to {context_data['absolute_end_line']})
2. Integrate trojan logic into the rare region lines
3. Provide ALL signal declarations with trojan_ prefix
4. Provide separate trigger/payload logic if needed
5. Maintain exact indentation and formatting
6. List all NEW trojan signals separately

=== OUTPUT FORMAT (JSON ONLY) ===
{{
  "signal_declarations": [
    "reg trojan_trigger;",
    "reg [31:0] trojan_leak_data;",
    "reg [7:0] trojan_shift_reg;"
  ],
  "new_trojan_signals": [
    "trojan_trigger",
    "trojan_leak_data", 
    "trojan_shift_reg"
  ],
  "modified_context_block": "COMPLETE code block with trojan integrated (ALL lines {context_data['absolute_start_line']}-{context_data['absolute_end_line']})",
  "trigger_logic": "additional always blocks for trigger detection (if needed outside context)",
  "payload_logic": "additional always blocks for payload (if needed outside context)",
  "integration_notes": "brief explanation of changes"
}}

=== VALIDATION CHECKLIST ===
Before outputting, verify:
□ All new signals use trojan_ prefix (no exceptions)
□ All new signals listed in new_trojan_signals array
□ All new signals declared in signal_declarations array
□ Only use signals from available signals list above
□ modified_context_block contains ALL {len(context_data['context_block'])} lines
□ No undefined signal references
□ No naming conflicts with existing signals

Output JSON now:"""
        
        return prompt, context_data
    
    def parse_improved_gpt_response(self, gpt_response: str, region_analysis: Dict, 
                                   context_data: Dict) -> Optional[Dict]:
        """Parse GPT response with context data."""
        response = gpt_response.strip()
        
        if '```json' in response:
            response = response.split('```json')[1].split('```')[0].strip()
        elif '```' in response:
            response = response.split('```')[1].split('```')[0].strip()
        
        try:
            impl_data = json.loads(response)
            
            implementation_id = f"HT{len(self.implementations) + 1:03d}"
            
            result = {
                'implementation_id': implementation_id,
                'region_id': region_analysis['region_id'],
                'chunk_id': region_analysis['chunk_id'],
                'ht_analysis': region_analysis['best_ht_match'],
                'rare_region_info': region_analysis['original_rare_region'],
                'context_info': {
                    'chunk_id': context_data['chunk_id'],
                    'context_start_line': context_data['absolute_start_line'],
                    'context_end_line': context_data['absolute_end_line'],
                    'rare_region_start_line': context_data['absolute_start_line'] + (context_data['rare_region_start_line'] - context_data['context_start_line']),
                    'rare_region_end_line': context_data['absolute_start_line'] + (context_data['rare_region_end_line'] - context_data['context_start_line']),
                    'total_context_lines': len(context_data['context_block'])
                },
                'implementation': {
                    'signal_declarations': impl_data.get('signal_declarations', []),
                    'new_trojan_signals': impl_data.get('new_trojan_signals', []),
                    'modified_context_block': impl_data.get('modified_context_block', ''),
                    'trigger_logic': impl_data.get('trigger_logic', ''),
                    'payload_logic': impl_data.get('payload_logic', ''),
                    'integration_notes': impl_data.get('integration_notes', '')
                }
            }
            
            self.implementations.append(result)
            return result
            
        except json.JSONDecodeError as e:
            print(f"JSON parse error: {e}")
            return None
    
    def run_improved_pipeline(self, ht_analysis_file: str, rare_regions_file: str, 
                             chunks_file: str, output_file: str, signals_file: str = None):
        """Run improved implementation pipeline with signal filtering."""
        print("HT IMPLEMENTATION WITH SIGNAL SCOPE VALIDATION")
        print("=" * 70)
        
        self.load_data(ht_analysis_file, rare_regions_file, chunks_file, signals_file)
        
        region_analyses = self.ht_data['trojan_analysis']['region_analysis']
        total = len(region_analyses)
        successful = 0
        
        for idx, region_analysis in enumerate(region_analyses, 1):
            if region_analysis.get('no_suitable_ht', False):
                print(f"\nSkipping {region_analysis['region_id']} (no suitable HT)")
                continue
            
            print(f"\n{'=' * 70}")
            print(f"IMPLEMENTATION {idx}/{total}: {region_analysis['region_id']}")
            print(f"{'=' * 70}")
            
            prompt, context_data = self.generate_improved_gpt_prompt(region_analysis)
            
            if not prompt:
                print(f"ERROR: Could not generate prompt for {region_analysis['region_id']}")
                continue
            
            print(f"\nGPT PROMPT:")
            print("─" * 70)
            print(prompt)
            print("─" * 70)
            
            print("\nPaste GPT response (type 'DONE' when finished, 'skip' to skip):")
            
            gpt_response = []
            while True:
                line = input()
                if line.strip() == 'DONE':
                    break
                if line.strip().lower() == 'skip':
                    gpt_response = None
                    break
                gpt_response.append(line)
            
            if gpt_response is None:
                print("Skipped")
                continue
            
            response_text = '\n'.join(gpt_response)
            impl_result = self.parse_improved_gpt_response(response_text, region_analysis, context_data)
            
            if impl_result:
                successful += 1
                new_signals = impl_result['implementation']['new_trojan_signals']
                print(f"\nSUCCESS: {impl_result['implementation_id']}")
                print(f"  New Trojan Signals: {', '.join(new_signals)}")
                print(f"  Context: Lines {impl_result['context_info']['context_start_line']}-{impl_result['context_info']['context_end_line']}")
            else:
                print("ERROR: Failed to parse response")
        
        results = {
            'metadata': {
                'total_regions': total,
                'successful_implementations': successful,
                'source_files': {
                    'ht_analysis': ht_analysis_file,
                    'rare_regions': rare_regions_file,
                    'chunks': chunks_file,
                    'signals': signals_file
                }
            },
            'implementations': self.implementations
        }
        
        with open(output_file, 'w') as f:
            json.dump(results, f, indent=2)
        
        print(f"\n{'=' * 70}")
        print(f"PIPELINE COMPLETE")
        print(f"Saved: {output_file}")
        print(f"Success: {successful}/{total}")
        
        return results


def run_improved_ht_implementation(ht_analysis_file, rare_regions_file, chunks_file, 
                                   signals_file=None, output_file=None):
    """Run improved implementation pipeline with signal scope validation."""
    if output_file is None:
        output_file = "/kaggle/working/RISCV_ht_implementations_v2.json"
    
    pipeline = ImprovedHTImplementationPipeline()
    return pipeline.run_improved_pipeline(
        ht_analysis_file, rare_regions_file, chunks_file, output_file, signals_file
    )


print("SIGNAL-AWARE HT IMPLEMENTATION READY")
print("="*60)
print("\nUsage:")
print("  run_improved_ht_implementation(")
print("      '/kaggle/working/RISCV_ht_analysis.json',")
print("      '/kaggle/working/RISCV_rare_regions.json',")
print("      '/kaggle/working/RISCV_chunks.json',")
print("      '/kaggle/working/RISCV_signals.json'")
print("  )")
print("\nSignal filtering: Only shows signals declared BEFORE context start line")

In [ ]:
import shutil

# List of source files
source_files = [
    "/kaggle/input/testing-data/RISCV_chunks.json",
    "/kaggle/input/testing-data/RISCV_ht_analysis.json",
    "/kaggle/input/testing-data/RISCV_rare_regions.json",
    "/kaggle/input/implementation/RISCV_ht_implementations_v2.json"
]

# Destination directory
dest_dir = "/kaggle/working/"

# Copy files
for file in source_files:
    shutil.copy(file, dest_dir)
    print(f"Copied {file} to {dest_dir}")


In [ ]:
run_improved_ht_implementation(
    '/kaggle/working/RISCV_ht_analysis.json',
    '/kaggle/working/RISCV_rare_regions.json',
    '/kaggle/working/RISCV_chunks.json',
    '/kaggle/working/RISCV_signals.json'
)

# GPT-Based Declaration Finder with Interactive Pipeline


In [1]:
import json
import os
import re
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

class GPTDeclarationFinder:
    """
    Interactive GPT-based declaration region finder.
    Groups trojans by module and gets optimal insertion points.
    """
    
    def __init__(self):
        self.declaration_points = {}  # module_id -> {line, reasoning, confidence}
    
    def load_data(self, implementations_file: str, chunks_file: str):
        """Load implementation and chunk data."""
        with open(implementations_file, 'r') as f:
            self.impl_data = json.load(f)
        
        with open(chunks_file, 'r') as f:
            self.chunks_data = json.load(f)
        
        print(f"✅ Loaded {len(self.impl_data['implementations'])} implementations")
        print(f"✅ Loaded {len(self.chunks_data['chunks'])} chunks")
    
    def group_trojans_by_module(self) -> Dict[str, List[Dict]]:
        """
        Group trojans by their parent module.
        Returns: {module_id: [implementations]}
        """
        module_groups = defaultdict(list)
        
        for impl in self.impl_data['implementations']:
            chunk_id = impl['chunk_id']
            
            # Extract module ID (base chunk without _a, _b suffix)
            parts = chunk_id.split('_')
            if len(parts) >= 2:
                module_id = f"{parts[0]}_{parts[1]}"
            else:
                module_id = chunk_id
            
            module_groups[module_id].append(impl)
        
        return dict(module_groups)
    
    def get_module_header_chunk(self, module_id: str) -> Tuple[str, Dict]:
        """
        Get the first chunk of a module (contains module declaration).
        Returns: (chunk_id, chunk_data)
        """
        # Find all chunks for this module
        module_chunks = []
        for chunk_id in sorted(self.chunks_data['chunks'].keys()):
            if chunk_id.startswith(module_id):
                module_chunks.append(chunk_id)
        
        if not module_chunks:
            return None, None
        
        # First chunk has the header
        first_chunk_id = module_chunks[0]
        return first_chunk_id, self.chunks_data['chunks'][first_chunk_id]
    
    def extract_module_name(self, chunk_content: str) -> str:
        """Extract module name from chunk content."""
        match = re.search(r'module\s+(\w+)', chunk_content)
        return match.group(1) if match else "unknown_module"
    
    def get_all_signal_declarations_needed(self, implementations: List[Dict]) -> List[str]:
        """
        Collect all unique signal declarations from multiple trojans in same module.
        """
        all_signals = []
        seen_signals = set()
        
        for impl in implementations:
            for sig_decl in impl['implementation']['signal_declarations']:
                sig_clean = sig_decl.strip()
                if sig_clean not in seen_signals:
                    all_signals.append(sig_decl)
                    seen_signals.add(sig_clean)
        
        return all_signals
    
    def generate_gpt_prompt_for_module(self, module_id: str, 
                                      implementations: List[Dict]) -> str:
        """
        Generate GPT prompt to find declaration insertion point.
        """
        # Get module header chunk
        header_chunk_id, header_chunk = self.get_module_header_chunk(module_id)
        
        if not header_chunk:
            return None
        
        module_name = self.extract_module_name(header_chunk['content'])
        chunk_lines = header_chunk['content'].split('\n')
        
        # Get all signals that need to be declared
        all_signals = self.get_all_signal_declarations_needed(implementations)
        
        # Build line-numbered code display
        line_display = []
        for i, line in enumerate(chunk_lines[:100], start=1):  # Show first 100 lines
            line_display.append(f"{i:3d}: {line}")
        
        prompt = f"""HARDWARE TROJAN DECLARATION INSERTION ANALYSIS

=== TASK ===
Analyze this SystemVerilog module and determine the OPTIMAL line number 
to insert hardware trojan signal declarations.

=== MODULE INFO ===
Module Name: {module_name}
Chunk ID: {header_chunk_id}
Starting Line: {header_chunk['start_line']}
Total Trojans in Module: {len(implementations)}

=== SIGNALS TO DECLARE ===
The following trojan signals need to be declared:
{chr(10).join([f"  - {sig}" for sig in all_signals])}

=== MODULE CODE (with line numbers) ===
{chr(10).join(line_display)}

=== INSERTION REQUIREMENTS ===
The declaration insertion point must be:
1. ✅ AFTER the module port list closing: );
2. ✅ AFTER all parameters and localparams
3. ✅ AFTER all typedef and enum declarations
4. ✅ AFTER all existing wire/reg/logic declarations (if any)
5. ✅ BEFORE any executable logic:
   - BEFORE assign statements
   - BEFORE always/always_ff/always_comb blocks
   - BEFORE initial blocks
   - BEFORE module instantiations
   - BEFORE generate blocks

=== EDGE CASES TO CONSIDER ===
- `ifdef / `ifndef blocks
- generate blocks
- Mixed parameter/signal declarations
- Comments between declarations
- Multiline declarations

=== OUTPUT FORMAT (JSON ONLY) ===
{{
  "insertion_line_in_chunk": 25,
  "absolute_line_number": {header_chunk['start_line'] + 25},
  "confidence": "high",
  "before_statement": "always_ff @(posedge clk)",
  "after_statement": "localparam DIVISOR = ...",
  "reasoning": "Insert after last localparam (line 24) and before first always_ff block (line 26). This ensures all declarations are complete before logic begins.",
  "verification": "Checked for ifdef blocks (none), generate blocks (none), and mixed declarations (none). Safe insertion point."
}}

=== CONFIDENCE LEVELS ===
- "high": Clear insertion point, no edge cases
- "medium": Some complexity but safe placement identified  
- "low": Complex structure, manual verification recommended

Output JSON now:"""
        
        return prompt
    
    def parse_gpt_declaration_response(self, gpt_response: str, 
                                      module_id: str) -> Optional[Dict]:
        """Parse GPT response for declaration insertion point."""
        response = gpt_response.strip()
        
        # Remove markdown
        if '```json' in response:
            response = response.split('```json')[1].split('```')[0].strip()
        elif '```' in response:
            response = response.split('```')[1].split('```')[0].strip()
        
        try:
            decl_info = json.loads(response)
            
            result = {
                'module_id': module_id,
                'insertion_line_in_chunk': decl_info['insertion_line_in_chunk'],
                'absolute_line_number': decl_info['absolute_line_number'],
                'confidence': decl_info['confidence'],
                'before_statement': decl_info.get('before_statement', 'N/A'),
                'after_statement': decl_info.get('after_statement', 'N/A'),
                'reasoning': decl_info['reasoning'],
                'verification': decl_info.get('verification', 'N/A')
            }
            
            self.declaration_points[module_id] = result
            return result
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON parse error: {e}")
            return None
    
    def run_declaration_finder(self, implementations_file: str, chunks_file: str,
                              output_file: str = None):
        """
        Main interactive pipeline to find declaration points.
        """
        print("="*70)
        print("🔍 GPT-BASED DECLARATION FINDER")
        print("="*70)
        
        # Load data
        self.load_data(implementations_file, chunks_file)
        
        # Group trojans by module
        module_groups = self.group_trojans_by_module()
        
        print(f"\n📊 Found {len(module_groups)} unique modules:")
        for module_id, impls in module_groups.items():
            print(f"   • {module_id}: {len(impls)} trojan(s)")
        
        print(f"\n{'='*70}")
        print("🎯 FINDING DECLARATION POINTS")
        print("="*70)
        print("\nFor each module, we'll ask GPT to identify the optimal")
        print("line number for inserting trojan signal declarations.\n")
        
        # Process each module
        for idx, (module_id, implementations) in enumerate(module_groups.items(), 1):
            print(f"\n{'='*70}")
            print(f"MODULE {idx}/{len(module_groups)}: {module_id}")
            print(f"{'='*70}")
            print(f"Trojans in this module: {len(implementations)}")
            
            trojan_ids = [impl['implementation_id'] for impl in implementations]
            print(f"Trojan IDs: {', '.join(trojan_ids)}")
            
            # Generate GPT prompt
            prompt = self.generate_gpt_prompt_for_module(module_id, implementations)
            
            if not prompt:
                print(f"❌ Could not generate prompt for {module_id}")
                continue
            
            print(f"\n{'─'*70}")
            print("📋 GPT PROMPT:")
            print("─"*70)
            print(prompt)
            print("─"*70)
            
            print("\n📝 INSTRUCTIONS:")
            print("1. Copy the prompt above")
            print("2. Paste into ChatGPT/Claude")
            print("3. Copy the JSON response")
            print("4. Paste below")
            print("Type 'skip' to skip, 'quit' to exit")
            
            while True:
                user_input = input(f"\nPaste declaration info for {module_id}: ").strip()
                
                if user_input.lower() == 'quit':
                    print("❌ Pipeline terminated")
                    return self.declaration_points
                
                if user_input.lower() == 'skip':
                    print("⏭️  Skipped")
                    break
                
                if len(user_input) < 20:
                    print("⚠️  Response too short. Paste complete JSON.")
                    continue
                
                # Parse response
                decl_result = self.parse_gpt_declaration_response(user_input, module_id)
                
                if decl_result:
                    print(f"\n✅ Declaration point saved for {module_id}:")
                    print(f"   Line: {decl_result['absolute_line_number']}")
                    print(f"   Confidence: {decl_result['confidence']}")
                    print(f"   After: {decl_result['after_statement']}")
                    print(f"   Before: {decl_result['before_statement']}")
                    break
                else:
                    print("❌ Parse failed. Try again or type 'skip'")
        
        # Save results
        if output_file is None:
            output_file = "/kaggle/working/declaration_points.json"
        
        output_data = {
            'metadata': {
                'total_modules': len(module_groups),
                'modules_processed': len(self.declaration_points)
            },
            'declaration_points': self.declaration_points
        }
        
        with open(output_file, 'w') as f:
            json.dump(output_data, f, indent=2)
        
        print(f"\n{'='*70}")
        print("✅ DECLARATION FINDER COMPLETE")
        print("="*70)
        print(f"Results saved: {output_file}")
        print(f"Modules processed: {len(self.declaration_points)}/{len(module_groups)}")
        
        return self.declaration_points
    
    def display_summary(self):
        """Display summary of found declaration points."""
        print("\n📊 DECLARATION POINTS SUMMARY")
        print("="*50)
        
        for module_id, info in self.declaration_points.items():
            print(f"\n📦 {module_id}:")
            print(f"   Line: {info['absolute_line_number']}")
            print(f"   Confidence: {info['confidence']}")
            print(f"   Reasoning: {info['reasoning'][:60]}...")


# ============================================================
# USAGE FUNCTION
# ============================================================

def find_declaration_points_with_gpt(implementations_file: str, 
                                     chunks_file: str,
                                     output_file: str = None):
    """
    Interactive GPT-based declaration point finder.
    
    Args:
        implementations_file: Path to RISCV_ht_implementations_v2.json
        chunks_file: Path to RISCV_chunks.json
        output_file: Where to save results (default: declaration_points.json)
    
    Returns:
        Dictionary of declaration points by module
    """
    finder = GPTDeclarationFinder()
    return finder.run_declaration_finder(
        implementations_file, 
        chunks_file, 
        output_file
    )


# ============================================================
# READY TO USE
# ============================================================

print("🎯 GPT-BASED DECLARATION FINDER READY!")
print("="*60)
print("\n📝 Usage:")
print("  declaration_points = find_declaration_points_with_gpt(")
print("      '/kaggle/working/RISCV_ht_implementations_v2.json',")
print("      '/kaggle/working/RISCV_chunks.json'")
print("  )")
print("\n✨ Features:")
print("  ✅ Groups trojans by module")
print("  ✅ One GPT call per module (not per trojan)")
print("  ✅ Shows all signals that need declaration")
print("  ✅ Line-numbered code display")
print("  ✅ Considers edge cases (ifdef, generate, etc.)")
print("  ✅ Saves results to JSON for later use")
print("\n⏱️  Typical time: ~5 minutes for 3-5 modules")
print("\n🚀 Run the function above to start!")

🎯 GPT-BASED DECLARATION FINDER READY!

📝 Usage:
  declaration_points = find_declaration_points_with_gpt(
      '/kaggle/working/RISCV_ht_implementations_v2.json',
      '/kaggle/working/RISCV_chunks.json'
  )

✨ Features:
  ✅ Groups trojans by module
  ✅ One GPT call per module (not per trojan)
  ✅ Shows all signals that need declaration
  ✅ Line-numbered code display
  ✅ Considers edge cases (ifdef, generate, etc.)
  ✅ Saves results to JSON for later use

⏱️  Typical time: ~5 minutes for 3-5 modules

🚀 Run the function above to start!


In [2]:
declaration_points = find_declaration_points_with_gpt(
      '/kaggle/working/RISCV_ht_implementations_v2.json',
      '/kaggle/working/RISCV_chunks.json'
  )


🔍 GPT-BASED DECLARATION FINDER
✅ Loaded 5 implementations
✅ Loaded 3 chunks

📊 Found 2 unique modules:
   • chunk_001: 3 trojan(s)
   • chunk_003: 2 trojan(s)

🎯 FINDING DECLARATION POINTS

For each module, we'll ask GPT to identify the optimal
line number for inserting trojan signal declarations.


MODULE 1/2: chunk_001
Trojans in this module: 3
Trojan IDs: HT001, HT002, HT003

──────────────────────────────────────────────────────────────────────
📋 GPT PROMPT:
──────────────────────────────────────────────────────────────────────
HARDWARE TROJAN DECLARATION INSERTION ANALYSIS

=== TASK ===
Analyze this SystemVerilog module and determine the OPTIMAL line number 
to insert hardware trojan signal declarations.

=== MODULE INFO ===
Module Name: uart_tx
Chunk ID: chunk_001
Starting Line: 1
Total Trojans in Module: 3

=== SIGNALS TO DECLARE ===
The following trojan signals need to be declared:
  - reg trojan_trigger;
  - reg [31:0] leak_data;
  - reg backdoor_enabled;
  - reg [3:0] state_r


Paste declaration info for chunk_001:  {   "insertion_line_in_chunk": 28,   "absolute_line_number": 29,   "confidence": "high",   "before_statement": "always_ff @(posedge clk or negedge rst_n) begin",   "after_statement": "logic baud_tick;",   "reasoning": "The last signal declaration is on line 27 ('logic baud_tick;'). The first executable statement ('always_ff @(posedge clk or negedge rst_n)') starts on line 29. Therefore, inserting the trojan signal declarations after line 27 and before line 29 keeps them grouped with other signal declarations and maintains syntactic correctness.",   "verification": "No `ifdef or generate blocks found. All parameter, typedef, and logic declarations occur before line 28, ensuring a clean separation between declarations and procedural logic. Safe insertion point." }



✅ Declaration point saved for chunk_001:
   Line: 29
   Confidence: high
   After: logic baud_tick;
   Before: always_ff @(posedge clk or negedge rst_n) begin

MODULE 2/2: chunk_003
Trojans in this module: 2
Trojan IDs: HT004, HT005

──────────────────────────────────────────────────────────────────────
📋 GPT PROMPT:
──────────────────────────────────────────────────────────────────────
HARDWARE TROJAN DECLARATION INSERTION ANALYSIS

=== TASK ===
Analyze this SystemVerilog module and determine the OPTIMAL line number 
to insert hardware trojan signal declarations.

=== MODULE INFO ===
Module Name: uart_tx_tb
Chunk ID: chunk_003
Starting Line: 144
Total Trojans in Module: 2

=== SIGNALS TO DECLARE ===
The following trojan signals need to be declared:
  - reg trojan_armed;
  - reg [7:0] stall_counter;
  - reg backdoor_enabled;
  - reg [3:0] state_reg;

=== MODULE CODE (with line numbers) ===
  1: module uart_tx_tb;
  2:     logic clk;
  3:     logic rst_n;
  4:     logic [7:0] tx_data;



Paste declaration info for chunk_003:  {   "insertion_line_in_chunk": 8,   "absolute_line_number": 152,   "confidence": "high",   "before_statement": "uart_tx #( .CLK_FREQ(1_000_000), .BAUD_RATE(9600) ) dut (.*);",   "after_statement": "logic tx_out;",   "reasoning": "The last signal declaration ends at line 7 ('logic tx_out;'). The next statement at line 9 is a module instantiation ('uart_tx ...'). To keep all declarations grouped together and before any procedural or instantiation logic, the optimal insertion point is immediately after line 7 and before line 9.",   "verification": "No parameters, typedefs, `ifdef blocks, or generate blocks detected. All existing signal declarations are contiguous (lines 2–7). Safe and syntactically correct insertion point." }



✅ Declaration point saved for chunk_003:
   Line: 152
   Confidence: high
   After: logic tx_out;
   Before: uart_tx #( .CLK_FREQ(1_000_000), .BAUD_RATE(9600) ) dut (.*);

✅ DECLARATION FINDER COMPLETE
Results saved: /kaggle/working/declaration_points.json
Modules processed: 2/2


# Complete Integration Pipeline 

In [5]:
import json
import os
import re
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

class ComprehensiveHTIntegrator:
    """
    Complete pipeline creating:
    1. Individual trojan modules (just the module)
    2. Complete RISCV files with SINGLE trojan each
    3. Complete RISCV file with ALL trojans combined
    """
    
    def __init__(self):
        self.output_dir = "/kaggle/working/final_trojan_files"
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(f"{self.output_dir}/individual_trojan_modules", exist_ok=True)
        os.makedirs(f"{self.output_dir}/complete_files_single_trojan", exist_ok=True)
        os.makedirs(f"{self.output_dir}/all_trojans_combined", exist_ok=True)
        self.declaration_points = {}
    
    def load_all_data(self, implementations_file: str, chunks_file: str, 
                     declaration_points_file: str):
        """Load implementation, chunk, and declaration data."""
        with open(implementations_file, 'r') as f:
            self.impl_data = json.load(f)
        
        with open(chunks_file, 'r') as f:
            self.chunks_data = json.load(f)
        
        with open(declaration_points_file, 'r') as f:
            decl_data = json.load(f)
            self.declaration_points = decl_data['declaration_points']
        
        print(f"✅ Loaded {len(self.impl_data['implementations'])} implementations")
        print(f"✅ Loaded {len(self.chunks_data['chunks'])} chunks")
        print(f"✅ Loaded {len(self.declaration_points)} declaration points")
    
    def get_module_id(self, chunk_id: str) -> str:
        """Extract module ID from chunk ID."""
        parts = chunk_id.split('_')
        if len(parts) >= 2:
            return f"{parts[0]}_{parts[1]}"
        return chunk_id
    
    def get_module_chunks(self, module_id: str) -> List[str]:
        """Get all chunk IDs for a module."""
        return sorted([cid for cid in self.chunks_data['chunks'].keys() 
                      if cid.startswith(module_id)])
    
    def integrate_trojan_into_module(self, implementation: Dict) -> str:
        """
        Integrate a single trojan into its module.
        Returns complete module code with trojan.
        """
        chunk_id = implementation['chunk_id']
        module_id = self.get_module_id(chunk_id)
        module_chunks = self.get_module_chunks(module_id)
        
        # Get declaration point
        decl_info = self.declaration_points.get(module_id)
        if not decl_info:
            print(f"  ⚠️  No declaration point for {module_id}")
            return None
        
        # Build module with trojan
        result_lines = []
        context_start = implementation['context_info']['context_start_line']
        context_end = implementation['context_info']['context_end_line']
        context_replaced = False
        
        for chunk_id in module_chunks:
            chunk = self.chunks_data['chunks'][chunk_id]
            chunk_start = chunk['start_line']
            chunk_lines = chunk['content'].split('\n')
            
            for line_idx, line in enumerate(chunk_lines):
                absolute_line = chunk_start + line_idx
                
                # Insert declarations
                if absolute_line == decl_info['absolute_line_number']:
                    result_lines.append("")
                    result_lines.append("    // ========== HARDWARE TROJAN DECLARATIONS ==========")
                    for sig_decl in implementation['implementation']['signal_declarations']:
                        result_lines.append(f"    {sig_decl}")
                    result_lines.append("    // ========== END TROJAN DECLARATIONS ==========")
                    result_lines.append("")
                
                # Replace context block
                if context_start <= absolute_line <= context_end:
                    if not context_replaced:
                        result_lines.append("    // ========== TROJAN-MODIFIED CONTEXT ==========")
                        modified_context = implementation['implementation']['modified_context_block']
                        
                        if '\\n' in modified_context:
                            modified_context = modified_context.replace('\\n', '\n')
                        
                        result_lines.append(modified_context)
                        result_lines.append("    // ========== END MODIFIED CONTEXT ==========")
                        
                        # Add trigger logic
                        if implementation['implementation'].get('trigger_logic'):
                            result_lines.append("")
                            result_lines.append("    // ========== TROJAN TRIGGER LOGIC ==========")
                            trigger = implementation['implementation']['trigger_logic']
                            if '\\n' in trigger:
                                trigger = trigger.replace('\\n', '\n')
                            result_lines.append(trigger)
                            result_lines.append("    // ========== END TRIGGER LOGIC ==========")
                        
                        # Add payload logic
                        if implementation['implementation'].get('payload_logic'):
                            result_lines.append("")
                            result_lines.append("    // ========== TROJAN PAYLOAD LOGIC ==========")
                            payload = implementation['implementation']['payload_logic']
                            if '\\n' in payload:
                                payload = payload.replace('\\n', '\n')
                            result_lines.append(payload)
                            result_lines.append("    // ========== END PAYLOAD LOGIC ==========")
                        
                        context_replaced = True
                    continue
                
                result_lines.append(line)
        
        return '\n'.join(result_lines)
    
    def create_individual_trojan_module(self, implementation: Dict, output_filename: str):
        """
        Create individual module file (JUST the trojaned module).
        """
        print(f"  📦 Creating module: {output_filename}")
        
        module_code = self.integrate_trojan_into_module(implementation)
        
        if not module_code:
            print(f"    ❌ Failed")
            return None
        
        filepath = os.path.join(self.output_dir, "individual_trojan_modules", output_filename)
        
        with open(filepath, 'w') as f:
            f.write(f"// Hardware Trojan Implementation: {implementation['implementation_id']}\n")
            f.write(f"// Region: {implementation['region_id']}\n")
            f.write(f"// HT Type: {implementation['ht_analysis']['trojan_type']}\n")
            f.write(f"// Stealth: {implementation['ht_analysis']['stealth_level']}\n")
            f.write(f"// Damage: {implementation['ht_analysis']['damage_potential']}\n")
            f.write(f"// Detection Difficulty: {implementation['ht_analysis']['detection_difficulty']}\n")
            f.write("\n")
            f.write(module_code)
        
        print(f"    ✅ Saved")
        return filepath
    
    def create_complete_file_with_single_trojan(self, implementation: Dict, output_filename: str):
        """
        Create COMPLETE RISCV.sv file with ONLY this one trojan integrated.
        All other modules remain original.
        """
        print(f"  📄 Creating complete file: {output_filename}")
        
        target_module_id = self.get_module_id(implementation['chunk_id'])
        
        # Get trojaned module code
        trojaned_module = self.integrate_trojan_into_module(implementation)
        
        if not trojaned_module:
            print(f"    ❌ Failed to integrate trojan")
            return None
        
        # Build complete file
        result_lines = []
        result_lines.append(f"// Complete RISC-V Implementation with Hardware Trojan {implementation['implementation_id']}")
        result_lines.append(f"// Trojan: {implementation['implementation_id']}")
        result_lines.append(f"// Region: {implementation['region_id']}")
        result_lines.append(f"// HT Type: {implementation['ht_analysis']['trojan_type']}")
        result_lines.append(f"// Stealth: {implementation['ht_analysis']['stealth_level']}")
        result_lines.append(f"// Damage: {implementation['ht_analysis']['damage_potential']}")
        result_lines.append("")
        
        # Get all unique modules in order
        processed_modules = set()
        
        for chunk_id in sorted(self.chunks_data['chunks'].keys()):
            module_id = self.get_module_id(chunk_id)
            
            # Skip if already processed
            if module_id in processed_modules:
                continue
            
            processed_modules.add(module_id)
            
            # If this is the target module, use trojaned version
            if module_id == target_module_id:
                result_lines.append(f"// ========== MODULE WITH TROJAN {implementation['implementation_id']} ==========")
                result_lines.append(trojaned_module)
                result_lines.append(f"// ========== END TROJANED MODULE ==========")
                result_lines.append("")
            else:
                # Use original module
                module_chunks = self.get_module_chunks(module_id)
                for chunk_id in module_chunks:
                    chunk = self.chunks_data['chunks'][chunk_id]
                    result_lines.append(chunk['content'])
                result_lines.append("")
        
        # Save file
        filepath = os.path.join(self.output_dir, "complete_files_single_trojan", output_filename)
        
        with open(filepath, 'w') as f:
            f.write('\n'.join(result_lines))
        
        print(f"    ✅ Saved")
        return filepath
    
    def create_all_trojans_combined_file(self):
        """
        Create ONE complete file with ALL trojans integrated.
        Handles multiple trojans per module.
        """
        print(f"\n{'='*70}")
        print("🔗 CREATING COMBINED FILE WITH ALL TROJANS")
        print("="*70)
        
        # Group implementations by module
        module_to_impls = defaultdict(list)
        for impl in self.impl_data['implementations']:
            module_id = self.get_module_id(impl['chunk_id'])
            module_to_impls[module_id].append(impl)
        
        # Check for conflicts
        conflicts = {k: v for k, v in module_to_impls.items() if len(v) > 1}
        if conflicts:
            print(f"\n⚠️  WARNING: {len(conflicts)} module(s) have multiple trojans")
            for module_id, impls in conflicts.items():
                impl_ids = [i['implementation_id'] for i in impls]
                print(f"  {module_id}: {', '.join(impl_ids)}")
            print("  Strategy: All trojans will be integrated sequentially")
        
        # Build complete file
        result_lines = []
        result_lines.append("// Complete RISC-V Implementation with ALL Hardware Trojans")
        result_lines.append(f"// Total Trojans: {len(self.impl_data['implementations'])}")
        result_lines.append(f"// Generated: {self.impl_data['metadata'].get('timestamp', 'unknown')}")
        result_lines.append("")
        
        trojan_list = []
        for impl in self.impl_data['implementations']:
            trojan_list.append(f"//   • {impl['implementation_id']}: {impl['ht_analysis']['trojan_type']} "
                              f"in {impl['region_id']} (Stealth: {impl['ht_analysis']['stealth_level']}, "
                              f"Damage: {impl['ht_analysis']['damage_potential']})")
        
        result_lines.extend(trojan_list)
        result_lines.append("")
        
        # Process each module
        processed_modules = set()
        
        for chunk_id in sorted(self.chunks_data['chunks'].keys()):
            module_id = self.get_module_id(chunk_id)
            
            if module_id in processed_modules:
                continue
            
            processed_modules.add(module_id)
            
            # Check if this module has trojans
            if module_id in module_to_impls:
                implementations = module_to_impls[module_id]
                print(f"  📦 {module_id}: Integrating {len(implementations)} trojan(s)")
                
                # Integrate all trojans for this module
                module_code = self.integrate_multiple_trojans_in_module(module_id, implementations)
                
                if module_code:
                    impl_ids = ', '.join([i['implementation_id'] for i in implementations])
                    result_lines.append(f"// ========== MODULE WITH TROJANS: {impl_ids} ==========")
                    result_lines.append(module_code)
                    result_lines.append(f"// ========== END TROJANED MODULE ==========")
                    result_lines.append("")
                else:
                    print(f"    ⚠️  Failed to integrate trojans")
            else:
                # Original module (no trojans)
                module_chunks = self.get_module_chunks(module_id)
                for chunk_id in module_chunks:
                    chunk = self.chunks_data['chunks'][chunk_id]
                    result_lines.append(chunk['content'])
                result_lines.append("")
        
        # Save file
        filepath = os.path.join(self.output_dir, "all_trojans_combined", "RISCV_ALL_TROJANS.sv")
        
        with open(filepath, 'w') as f:
            f.write('\n'.join(result_lines))
        
        print(f"\n  ✅ Saved: {filepath}")
        return filepath
    
    def integrate_multiple_trojans_in_module(self, module_id: str, 
                                            implementations: List[Dict]) -> str:
        """
        Integrate multiple trojans into one module.
        Merges declarations, applies context modifications sequentially.
        """
        module_chunks = self.get_module_chunks(module_id)
        decl_info = self.declaration_points.get(module_id)
        
        if not decl_info:
            return None
        
        # Collect all signal declarations (deduplicate)
        all_signals = []
        seen_signals = set()
        for impl in implementations:
            for sig_decl in impl['implementation']['signal_declarations']:
                sig_clean = sig_decl.strip()
                if sig_clean not in seen_signals:
                    all_signals.append(sig_decl)
                    seen_signals.add(sig_clean)
        
        # Build list of context replacements
        context_replacements = []
        for impl in implementations:
            context_replacements.append({
                'start': impl['context_info']['context_start_line'],
                'end': impl['context_info']['context_end_line'],
                'implementation': impl
            })
        
        # Sort by start line
        context_replacements.sort(key=lambda x: x['start'])
        
        # Build module
        result_lines = []
        skip_until = -1
        
        for chunk_id in module_chunks:
            chunk = self.chunks_data['chunks'][chunk_id]
            chunk_start = chunk['start_line']
            chunk_lines = chunk['content'].split('\n')
            
            for line_idx, line in enumerate(chunk_lines):
                absolute_line = chunk_start + line_idx
                
                # Skip lines in replaced contexts
                if absolute_line <= skip_until:
                    continue
                
                # Insert all declarations at once
                if absolute_line == decl_info['absolute_line_number']:
                    result_lines.append("")
                    result_lines.append("    // ========== HARDWARE TROJAN DECLARATIONS (ALL) ==========")
                    for sig_decl in all_signals:
                        result_lines.append(f"    {sig_decl}")
                    result_lines.append("    // ========== END TROJAN DECLARATIONS ==========")
                    result_lines.append("")
                
                # Check if this line is in any context replacement
                in_context = False
                for ctx in context_replacements:
                    if ctx['start'] <= absolute_line <= ctx['end']:
                        in_context = True
                        impl = ctx['implementation']
                        
                        # Insert modified context
                        result_lines.append(f"    // === TROJAN {impl['implementation_id']}: {impl['region_id']} ===")
                        modified = impl['implementation']['modified_context_block']
                        if '\\n' in modified:
                            modified = modified.replace('\\n', '\n')
                        result_lines.append(modified)
                        
                        # Add trigger/payload
                        if impl['implementation'].get('trigger_logic'):
                            result_lines.append("")
                            result_lines.append(f"    // === TRIGGER: {impl['implementation_id']} ===")
                            trigger = impl['implementation']['trigger_logic']
                            if '\\n' in trigger:
                                trigger = trigger.replace('\\n', '\n')
                            result_lines.append(trigger)
                        
                        if impl['implementation'].get('payload_logic'):
                            result_lines.append("")
                            result_lines.append(f"    // === PAYLOAD: {impl['implementation_id']} ===")
                            payload = impl['implementation']['payload_logic']
                            if '\\n' in payload:
                                payload = payload.replace('\\n', '\n')
                            result_lines.append(payload)
                        
                        result_lines.append(f"    // === END TROJAN {impl['implementation_id']} ===")
                        result_lines.append("")
                        
                        skip_until = ctx['end']
                        break
                
                if not in_context:
                    result_lines.append(line)
        
        return '\n'.join(result_lines)
    
    def create_original_file(self):
        """Create original file (no trojans)."""
        print(f"\n📄 Creating original file...")
        
        lines = []
        lines.append("// Original RISC-V Design (No Trojans)")
        lines.append(f"// Source: {self.chunks_data['file_metadata']['filename']}")
        lines.append("")
        
        for chunk_id in sorted(self.chunks_data['chunks'].keys()):
            chunk = self.chunks_data['chunks'][chunk_id]
            lines.append(chunk['content'])
            lines.append("")
        
        filepath = os.path.join(self.output_dir, "RISCV_ORIGINAL.sv")
        with open(filepath, 'w') as f:
            f.write('\n'.join(lines))
        
        print(f"  ✅ Saved: {filepath}")
        return filepath
    
    def create_summary_report(self):
        """Create comprehensive summary JSON."""
        print(f"\n📊 Creating summary report...")
        
        summary = {
            "metadata": {
                "total_implementations": len(self.impl_data['implementations']),
                "source_file": self.chunks_data['file_metadata']['filename'],
                "output_directory": self.output_dir
            },
            "files_generated": {
                "original": "RISCV_ORIGINAL.sv",
                "individual_modules": [],
                "complete_single_trojan": [],
                "all_trojans_combined": "all_trojans_combined/RISCV_ALL_TROJANS.sv"
            },
            "trojan_details": []
        }
        
        for impl in self.impl_data['implementations']:
            impl_id = impl['implementation_id']
            ht_type = impl['ht_analysis']['trojan_type']
            region_id = impl['region_id']
            
            module_filename = f"{impl_id}_{region_id}_{ht_type}.sv"
            complete_filename = f"RISCV_with_{impl_id}.sv"
            
            summary['files_generated']['individual_modules'].append(
                f"individual_trojan_modules/{module_filename}"
            )
            summary['files_generated']['complete_single_trojan'].append(
                f"complete_files_single_trojan/{complete_filename}"
            )
            
            summary['trojan_details'].append({
                "id": impl_id,
                "region": region_id,
                "chunk": impl['chunk_id'],
                "type": ht_type,
                "stealth": impl['ht_analysis']['stealth_level'],
                "damage": impl['ht_analysis']['damage_potential'],
                "detection_difficulty": impl['ht_analysis']['detection_difficulty'],
                "context_lines": f"{impl['context_info']['context_start_line']}-{impl['context_info']['context_end_line']}",
                "individual_module_file": f"individual_trojan_modules/{module_filename}",
                "complete_file": f"complete_files_single_trojan/{complete_filename}"
            })
        
        filepath = os.path.join(self.output_dir, "trojan_summary.json")
        with open(filepath, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"  ✅ Saved: {filepath}")
        return filepath
    
    def run_complete_pipeline(self, implementations_file: str, chunks_file: str,
                             declaration_points_file: str):
        """Run complete integration pipeline."""
        print("="*70)
        print("🚀 COMPREHENSIVE HT INTEGRATION PIPELINE")
        print("="*70)
        
        # Load data
        self.load_all_data(implementations_file, chunks_file, declaration_points_file)
        
        # Create original file
        self.create_original_file()
        
        # Process each trojan
        print(f"\n{'='*70}")
        print("📦 CREATING FILES FOR EACH TROJAN")
        print("="*70)
        
        for idx, impl in enumerate(self.impl_data['implementations'], 1):
            impl_id = impl['implementation_id']
            ht_type = impl['ht_analysis']['trojan_type']
            region_id = impl['region_id']
            
            print(f"\n[{idx}/{len(self.impl_data['implementations'])}] Processing {impl_id}...")
            
            # 1. Individual module file
            module_filename = f"{impl_id}_{region_id}_{ht_type}.sv"
            self.create_individual_trojan_module(impl, module_filename)
            
            # 2. Complete file with single trojan
            complete_filename = f"RISCV_with_{impl_id}.sv"
            self.create_complete_file_with_single_trojan(impl, complete_filename)
        
        # Create combined file
        self.create_all_trojans_combined_file()
        
        # Create summary
        self.create_summary_report()
        
        # Final summary
        print(f"\n{'='*70}")
        print("✅ PIPELINE COMPLETE")
        print("="*70)
        print(f"\n📁 Output Directory: {self.output_dir}")
        print(f"\n📂 Generated Files:")
        print(f"   ✅ 1 original file: RISCV_ORIGINAL.sv")
        print(f"   ✅ {len(self.impl_data['implementations'])} individual module files")
        print(f"   ✅ {len(self.impl_data['implementations'])} complete single-trojan files")
        print(f"   ✅ 1 combined file: RISCV_ALL_TROJANS.sv")
        print(f"   ✅ 1 summary: trojan_summary.json")
        print(f"\n📊 Total: {3 + 2*len(self.impl_data['implementations'])} files")
        print("="*70)


def integrate_all_trojans_complete(
    implementations_file: str, 
    chunks_file: str,
    declaration_points_file: str
):
    """
    Complete integration creating ALL file variants.
    """
    integrator = ComprehensiveHTIntegrator()
    integrator.run_complete_pipeline(
        implementations_file, 
        chunks_file, 
        declaration_points_file
    )


print("🎯 COMPLETE INTEGRATION PIPELINE READY!")
print("="*60)
print("\n✨ This creates:")
print("  📄 RISCV_ORIGINAL.sv")
print("  📦 individual_trojan_modules/ (just the module)")
print("  📄 complete_files_single_trojan/ (full RISCV with 1 trojan)")
print("  📄 all_trojans_combined/RISCV_ALL_TROJANS.sv")
print("  📊 trojan_summary.json")
print("\n🚀 Usage:")
print("  integrate_all_trojans_complete(")
print("      '/kaggle/working/RISCV_ht_implementations_v2.json',")
print("      '/kaggle/working/RISCV_chunks.json',")
print("      '/kaggle/working/declaration_points.json'")
print("  )")

🎯 COMPLETE INTEGRATION PIPELINE READY!

✨ This creates:
  📄 RISCV_ORIGINAL.sv
  📦 individual_trojan_modules/ (just the module)
  📄 complete_files_single_trojan/ (full RISCV with 1 trojan)
  📄 all_trojans_combined/RISCV_ALL_TROJANS.sv
  📊 trojan_summary.json

🚀 Usage:
  integrate_all_trojans_complete(
      '/kaggle/working/RISCV_ht_implementations_v2.json',
      '/kaggle/working/RISCV_chunks.json',
      '/kaggle/working/declaration_points.json'
  )


In [6]:
integrate_all_trojans_with_gpt_declarations(
      'RISCV_ht_implementations_v2.json',
      'RISCV_chunks.json',
      'declaration_points.json'
  )

🚀 COMPREHENSIVE HT INTEGRATION PIPELINE
✅ Loaded 5 implementations
✅ Loaded 3 chunks
✅ Loaded 2 declaration points

📄 Creating original file...
  ✅ Saved: /kaggle/working/final_trojan_files/RISCV_ORIGINAL.sv

📦 CREATING FILES FOR EACH TROJAN

[1/5] Processing HT001...
  📦 Creating module: HT001_chunk_001_RESET_SEQUENCE_HT2_Leakage.sv
    ✅ Saved
  📄 Creating complete file: RISCV_with_HT001.sv
    ✅ Saved

[2/5] Processing HT002...
  📦 Creating module: HT002_chunk_001_RESET_SEQUENCE_HT1_Privilege.sv
    ✅ Saved
  📄 Creating complete file: RISCV_with_HT002.sv
    ✅ Saved

[3/5] Processing HT003...
  📦 Creating module: HT003_chunk_001_RARE_DEFAULT_RECOVERY_HT2_Leakage.sv
    ✅ Saved
  📄 Creating complete file: RISCV_with_HT003.sv
    ✅ Saved

[4/5] Processing HT004...
  📦 Creating module: HT004_chunk_003_WAIT_FOR_READY_SUSPENSION_HT3_Performance.sv
    ✅ Saved
  📄 Creating complete file: RISCV_with_HT004.sv
    ✅ Saved

[5/5] Processing HT005...
  📦 Creating module: HT005_chunk_003_DEBUG_

# FILE STRUCTURE DESCRIPTION

In [7]:
import os
import json
from pathlib import Path
from datetime import datetime

class TrojanFileStructureViewer:
    """
    Display complete file structure with descriptions and metadata.
    """
    
    def __init__(self, base_dir: str = "/kaggle/working"):
        self.base_dir = base_dir
        self.tree_lines = []
        
    def get_file_size(self, filepath: str) -> str:
        """Get human-readable file size."""
        try:
            size = os.path.getsize(filepath)
            for unit in ['B', 'KB', 'MB', 'GB']:
                if size < 1024.0:
                    return f"{size:.1f} {unit}"
                size /= 1024.0
        except:
            return "N/A"
    
    def get_file_description(self, filename: str, filepath: str) -> str:
        """Get description for each file type."""
        descriptions = {
            'declaration_points.json': 'GPT-identified declaration insertion points for each module',
            'RISCV_ORIGINAL.sv': 'Clean baseline design without any trojans (reference)',
            'trojan_summary.json': 'Complete summary of all trojans with metadata and file paths',
            'RISCV_ALL_TROJANS.sv': 'Complete RISC-V design with ALL trojans integrated',
        }
        
        # Individual module descriptions
        if filename.startswith('HT') and 'individual_trojan_modules' in filepath:
            return 'Isolated trojaned module (module code only, not full design)'
        
        # Complete single trojan files
        if filename.startswith('RISCV_with_HT') and 'complete_files_single_trojan' in filepath:
            ht_id = filename.replace('RISCV_with_', '').replace('.sv', '')
            return f'Complete RISC-V design with ONLY {ht_id} integrated (compilable)'
        
        return descriptions.get(filename, '')
    
    def get_directory_description(self, dirname: str) -> str:
        """Get description for directories."""
        descriptions = {
            'final_trojan_files': 'Main output directory containing all generated files',
            'individual_trojan_modules': 'Individual trojaned modules (module code only)',
            'complete_files_single_trojan': 'Complete RISC-V files with ONE trojan each',
            'all_trojans_combined': 'Complete RISC-V file with ALL trojans together'
        }
        return descriptions.get(dirname, '')
    
    def count_files_in_dir(self, dirpath: str) -> int:
        """Count files in directory."""
        try:
            return len([f for f in os.listdir(dirpath) if os.path.isfile(os.path.join(dirpath, f))])
        except:
            return 0
    
    def print_structure(self):
        """Print complete file structure with descriptions."""
        print("=" * 90)
        print("📁 HARDWARE TROJAN FILE STRUCTURE")
        print("=" * 90)
        print()
        
        # Check if main directories exist
        final_dir = os.path.join(self.base_dir, "final_trojan_files")
        decl_file = os.path.join(self.base_dir, "declaration_points.json")
        
        if not os.path.exists(final_dir) and not os.path.exists(decl_file):
            print("⚠️  WARNING: Output directories not found!")
            print(f"   Expected location: {self.base_dir}")
            print("   Run the integration pipeline first.")
            return
        
        print(f"📂 Base Directory: {self.base_dir}")
        print()
        
        # Root level files
        self._print_root_level()
        
        # Main output directory
        if os.path.exists(final_dir):
            self._print_final_trojan_files()
        
        # Summary statistics
        print()
        self._print_summary()
        
        print()
        print("=" * 90)
    
    def _print_root_level(self):
        """Print root level files."""
        decl_file = os.path.join(self.base_dir, "declaration_points.json")
        
        print("📄 declaration_points.json", end="")
        if os.path.exists(decl_file):
            size = self.get_file_size(decl_file)
            desc = self.get_file_description('declaration_points.json', decl_file)
            print(f"  [{size}]")
            print(f"   └─ {desc}")
        else:
            print("  [NOT FOUND]")
        print()
    
    def _print_final_trojan_files(self):
        """Print final_trojan_files directory structure."""
        final_dir = os.path.join(self.base_dir, "final_trojan_files")
        
        print("📁 final_trojan_files/")
        desc = self.get_directory_description('final_trojan_files')
        print(f"   └─ {desc}")
        print()
        
        # Top-level files in final_trojan_files
        self._print_top_level_files(final_dir)
        
        # Subdirectories
        self._print_individual_modules(final_dir)
        self._print_complete_single_trojan(final_dir)
        self._print_all_trojans_combined(final_dir)
    
    def _print_top_level_files(self, final_dir: str):
        """Print top-level files in final_trojan_files."""
        files_to_check = ['RISCV_ORIGINAL.sv', 'trojan_summary.json']
        
        for filename in files_to_check:
            filepath = os.path.join(final_dir, filename)
            
            print(f"   ├─ 📄 {filename}", end="")
            
            if os.path.exists(filepath):
                size = self.get_file_size(filepath)
                desc = self.get_file_description(filename, filepath)
                print(f"  [{size}]")
                print(f"   │     └─ {desc}")
            else:
                print("  [NOT FOUND]")
            
            print("   │")
    
    def _print_individual_modules(self, final_dir: str):
        """Print individual_trojan_modules directory."""
        modules_dir = os.path.join(final_dir, "individual_trojan_modules")
        
        print("   ├─ 📁 individual_trojan_modules/")
        desc = self.get_directory_description('individual_trojan_modules')
        print(f"   │     └─ {desc}")
        
        if os.path.exists(modules_dir):
            files = sorted([f for f in os.listdir(modules_dir) if f.endswith('.sv')])
            
            if files:
                for idx, filename in enumerate(files):
                    filepath = os.path.join(modules_dir, filename)
                    size = self.get_file_size(filepath)
                    
                    is_last = (idx == len(files) - 1)
                    prefix = "└─" if is_last else "├─"
                    
                    print(f"   │     {prefix} {filename}  [{size}]")
                    
                    # Extract trojan info from filename
                    if filename.startswith('HT'):
                        parts = filename.replace('.sv', '').split('_')
                        if len(parts) >= 3:
                            ht_id = parts[0]
                            ht_type = parts[-1]
                            desc = f"Module with {ht_id} trojan (Type: {ht_type})"
                            connector = "   " if is_last else "│  "
                            print(f"   │     {connector}   └─ {desc}")
            else:
                print("   │        └─ [No files found]")
        else:
            print("   │        └─ [Directory not found]")
        
        print("   │")
    
    def _print_complete_single_trojan(self, final_dir: str):
        """Print complete_files_single_trojan directory."""
        complete_dir = os.path.join(final_dir, "complete_files_single_trojan")
        
        print("   ├─ 📁 complete_files_single_trojan/")
        desc = self.get_directory_description('complete_files_single_trojan')
        print(f"   │     └─ {desc}")
        
        if os.path.exists(complete_dir):
            files = sorted([f for f in os.listdir(complete_dir) if f.endswith('.sv')])
            
            if files:
                for idx, filename in enumerate(files):
                    filepath = os.path.join(complete_dir, filename)
                    size = self.get_file_size(filepath)
                    
                    is_last = (idx == len(files) - 1)
                    prefix = "└─" if is_last else "├─"
                    
                    print(f"   │     {prefix} {filename}  [{size}]")
                    
                    # Extract HT ID
                    if 'HT' in filename:
                        ht_id = filename.replace('RISCV_with_', '').replace('.sv', '')
                        desc = f"Complete RISC-V design with ONLY {ht_id} integrated ✅ Compilable"
                        connector = "   " if is_last else "│  "
                        print(f"   │     {connector}   └─ {desc}")
            else:
                print("   │        └─ [No files found]")
        else:
            print("   │        └─ [Directory not found]")
        
        print("   │")
    
    def _print_all_trojans_combined(self, final_dir: str):
        """Print all_trojans_combined directory."""
        combined_dir = os.path.join(final_dir, "all_trojans_combined")
        
        print("   └─ 📁 all_trojans_combined/")
        desc = self.get_directory_description('all_trojans_combined')
        print(f"         └─ {desc}")
        
        if os.path.exists(combined_dir):
            combined_file = os.path.join(combined_dir, "RISCV_ALL_TROJANS.sv")
            
            if os.path.exists(combined_file):
                size = self.get_file_size(combined_file)
                print(f"         └─ RISCV_ALL_TROJANS.sv  [{size}]")
                print(f"               └─ Complete RISC-V with ALL trojans integrated ✅ Compilable")
            else:
                print("            └─ [File not found]")
        else:
            print("            └─ [Directory not found]")
    
    def _print_summary(self):
        """Print summary statistics."""
        final_dir = os.path.join(self.base_dir, "final_trojan_files")
        
        print("=" * 90)
        print("📊 SUMMARY STATISTICS")
        print("=" * 90)
        
        if not os.path.exists(final_dir):
            print("⚠️  No files generated yet")
            return
        
        # Count files
        modules_dir = os.path.join(final_dir, "individual_trojan_modules")
        complete_dir = os.path.join(final_dir, "complete_files_single_trojan")
        combined_dir = os.path.join(final_dir, "all_trojans_combined")
        
        module_count = self.count_files_in_dir(modules_dir)
        complete_count = self.count_files_in_dir(complete_dir)
        combined_count = self.count_files_in_dir(combined_dir)
        
        print(f"📄 Original File:              1 file")
        print(f"📦 Individual Modules:         {module_count} files")
        print(f"📄 Complete Single Trojan:     {complete_count} files")
        print(f"📄 All Trojans Combined:       {combined_count} file")
        print(f"📊 Summary JSON:               1 file")
        print(f"─" * 90)
        print(f"✅ TOTAL FILES:                {3 + module_count + complete_count + combined_count} files")
        
        # Read summary if available
        summary_file = os.path.join(final_dir, "trojan_summary.json")
        if os.path.exists(summary_file):
            try:
                with open(summary_file, 'r') as f:
                    summary = json.load(f)
                
                print()
                print("🎯 TROJAN DETAILS:")
                print("─" * 90)
                
                for trojan in summary.get('trojan_details', []):
                    print(f"   • {trojan['id']}: {trojan['type']} in {trojan['region']}")
                    print(f"     Stealth: {trojan['stealth']} | Damage: {trojan['damage']} | Detection: {trojan['detection_difficulty']}")
                
            except:
                pass
        
        print()
        print("💡 FILE USAGE GUIDE:")
        print("─" * 90)
        print("   📄 RISCV_ORIGINAL.sv          → Baseline for comparison (clean design)")
        print("   📦 individual_trojan_modules/ → Study trojan logic in isolation")
        print("   📄 complete_files_single/     → Test each trojan separately (compilable)")
        print("   📄 RISCV_ALL_TROJANS.sv       → Test all trojans together (compilable)")
        print("   📊 trojan_summary.json        → Metadata and file mappings")


def display_file_structure(base_dir: str = "/kaggle/working"):
    """
    Display complete file structure with descriptions.
    
    Args:
        base_dir: Base directory to scan (default: /kaggle/working)
    """
    viewer = TrojanFileStructureViewer(base_dir)
    viewer.print_structure()


def display_structure_compact():
    """Display compact version of file structure."""
    print("=" * 70)
    print("📁 QUICK FILE STRUCTURE REFERENCE")
    print("=" * 70)
    print("""
/kaggle/working/
│
├── 📄 declaration_points.json                    # GPT insertion points
│
└── 📁 final_trojan_files/
    │
    ├── 📄 RISCV_ORIGINAL.sv                      # Clean baseline
    ├── 📄 trojan_summary.json                    # Summary metadata
    │
    ├── 📁 individual_trojan_modules/             # Module code only
    │   ├── HT001_*.sv
    │   ├── HT002_*.sv
    │   └── ...
    │
    ├── 📁 complete_files_single_trojan/          # Full design, 1 trojan
    │   ├── RISCV_with_HT001.sv                   # ✅ Compilable
    │   ├── RISCV_with_HT002.sv                   # ✅ Compilable
    │   └── ...
    │
    └── 📁 all_trojans_combined/
        └── RISCV_ALL_TROJANS.sv                  # ✅ Compilable (all)
    """)
    print("=" * 70)


def compare_file_sizes():
    """Compare file sizes across different variants."""
    base_dir = "/kaggle/working/final_trojan_files"
    
    if not os.path.exists(base_dir):
        print("⚠️  Files not generated yet")
        return
    
    print("=" * 70)
    print("📊 FILE SIZE COMPARISON")
    print("=" * 70)
    print()
    
    original = os.path.join(base_dir, "RISCV_ORIGINAL.sv")
    combined = os.path.join(base_dir, "all_trojans_combined", "RISCV_ALL_TROJANS.sv")
    
    if os.path.exists(original):
        orig_size = os.path.getsize(original)
        print(f"📄 Original Design:        {orig_size:>10,} bytes")
    
    if os.path.exists(combined):
        comb_size = os.path.getsize(combined)
        print(f"📄 All Trojans Combined:   {comb_size:>10,} bytes")
        
        if os.path.exists(original):
            overhead = comb_size - orig_size
            percentage = (overhead / orig_size) * 100
            print(f"📊 Trojan Overhead:        {overhead:>10,} bytes (+{percentage:.1f}%)")
    
    print()
    
    # Individual complete files
    complete_dir = os.path.join(base_dir, "complete_files_single_trojan")
    if os.path.exists(complete_dir):
        files = sorted([f for f in os.listdir(complete_dir) if f.endswith('.sv')])
        
        if files:
            print("📄 Individual Trojan Variants:")
            print("─" * 70)
            
            for filename in files:
                filepath = os.path.join(complete_dir, filename)
                size = os.path.getsize(filepath)
                ht_id = filename.replace('RISCV_with_', '').replace('.sv', '')
                print(f"   {ht_id:12s}  {size:>10,} bytes")
    
    print("=" * 70)


# Ready to use
print("🎯 FILE STRUCTURE VIEWER READY!")
print("=" * 60)
print("\n📝 Usage:")
print("  1. display_file_structure()          # Detailed structure with descriptions")
print("  2. display_structure_compact()       # Quick reference (ASCII tree)")
print("  3. compare_file_sizes()              # File size comparison")
print("\n🚀 Run any function above to view the structure!")

🎯 FILE STRUCTURE VIEWER READY!

📝 Usage:
  1. display_file_structure()          # Detailed structure with descriptions
  2. display_structure_compact()       # Quick reference (ASCII tree)
  3. compare_file_sizes()              # File size comparison

🚀 Run any function above to view the structure!


## FILE STRUCTURE

In [8]:
display_file_structure()          # Detailed structure with descriptions
display_structure_compact()       # Quick reference (ASCII tree)
compare_file_sizes()              # File size comparison

📁 HARDWARE TROJAN FILE STRUCTURE

📂 Base Directory: /kaggle/working

📄 declaration_points.json  [1.7 KB]
   └─ GPT-identified declaration insertion points for each module

📁 final_trojan_files/
   └─ Main output directory containing all generated files

   ├─ 📄 RISCV_ORIGINAL.sv  [4.4 KB]
   │     └─ Clean baseline design without any trojans (reference)
   │
   ├─ 📄 trojan_summary.json  [3.3 KB]
   │     └─ Complete summary of all trojans with metadata and file paths
   │
   ├─ 📁 individual_trojan_modules/
   │     └─ Individual trojaned modules (module code only)
   │     ├─ HT001_chunk_001_RESET_SEQUENCE_HT2_Leakage.sv  [4.9 KB]
   │     │     └─ Module with HT001 trojan (Type: Leakage)
   │     ├─ HT002_chunk_001_RESET_SEQUENCE_HT1_Privilege.sv  [5.1 KB]
   │     │     └─ Module with HT002 trojan (Type: Privilege)
   │     ├─ HT003_chunk_001_RARE_DEFAULT_RECOVERY_HT2_Leakage.sv  [5.0 KB]
   │     │     └─ Module with HT003 trojan (Type: Leakage)
   │     ├─ HT004_chunk_003_WAIT_FOR_